In [ ]:
from pathlib import Path
import sys

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

print("Python:", sys.executable)
print("Diretório atual:", Path.cwd())
print("Repositório existe:", REPO.is_dir())

for pasta in [
    REPO,
    REPO / "data",
    REPO / "datasets",
]:
    print(f"\n--- {pasta} ---")

    if not pasta.is_dir():
        print("Pasta não encontrada")
        continue

    itens = sorted(pasta.iterdir(), key=lambda p: p.name)

    for item in itens[:40]:
        tipo = "[DIR]" if item.is_dir() else "[FILE]"
        print(tipo, item.name)

    if len(itens) > 40:
        print(f"... mais {len(itens) - 40} itens")

In [ ]:
from pathlib import Path

# 1. Listar os dataloaders existentes
pasta = REPO / "dataloaders"

print("=== DATALOADERS ===")
for arquivo in sorted(pasta.iterdir()):
    print(arquivo.name)

# 2. Procurar referências a CT e MRI no código
termos = ["chaos", "sabs", "synapse", "mri"]

print("\n=== REFERÊNCIAS AOS DATASETS ===")

arquivos = [
    REPO / "README.md",
    *pasta.rglob("*.py"),
]

for arquivo in arquivos:
    linhas = arquivo.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()

    encontrados = [
        (i, linha.strip())
        for i, linha in enumerate(linhas, 1)
        if any(t in linha.lower() for t in termos)
    ]

    if encontrados:
        print(f"\n{arquivo.relative_to(REPO)}")
        for numero, linha in encontrados[:15]:
            print(f"  L{numero}: {linha[:180]}")

In [ ]:
from pathlib import Path

arquivos = {
    "README - Instruções de datasets":
        (REPO / "README.md", 1, 28),

    "Dataset Utils - Configuração":
        (REPO / "dataloaders/dataset_utils.py", 1, 75),
}

for titulo, (caminho, inicio, fim) in arquivos.items():
    print(f"\n{'=' * 60}")
    print(titulo)
    print(f"{'=' * 60}")

    linhas = caminho.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for numero in range(inicio, min(fim, len(linhas)) + 1):
        print(f"{numero:3}: {linhas[numero - 1]}")

In [ ]:
import json

notebook_path = REPO / "data" / "data_processing.ipynb"

if not notebook_path.exists():
    print("Notebook não encontrado!")
else:
    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    print("Total de células:", len(notebook["cells"]))

    termos = [
        "sabs", "chaos", "download",
        "nifti", "dicom", "superpix",
        "felzenszwalb", "data_dir"
    ]

    for i, cell in enumerate(notebook["cells"]):
        source = "".join(cell.get("source", []))

        if any(t in source.lower() for t in termos):
            print(f"\n{'=' * 50}")
            print(f"Célula {i} ({cell['cell_type']})")
            print(f"{'=' * 50}")
            print(source[:1200])

In [ ]:
from pathlib import Path
import shutil

pastas = [
    REPO.parent,
    REPO / "data",
    Path("/home/1542521"),
]

for pasta in pastas:
    print(f"\n=== {pasta} ===")

    if not pasta.is_dir():
        print("Não encontrada")
        continue

    diretorios = sorted(
        [p for p in pasta.iterdir() if p.is_dir()],
        key=lambda p: p.name.lower()
    )

    for diretorio in diretorios[:60]:
        print("📁", diretorio.name)

    if len(diretorios) > 60:
        print(f"... outros {len(diretorios) - 60} diretórios")

total, usado, livre = shutil.disk_usage(REPO)

print("\n=== ESPAÇO EM DISCO ===")
print(f"Total: {total / 1024**3:.1f} GB")
print(f"Usado: {usado / 1024**3:.1f} GB")
print(f"Livre: {livre / 1024**3:.1f} GB")

In [ ]:
from pathlib import Path

ORIGINAL = Path(
    "/home/1542521/cassio_pereira/ProtoSAM"
)

pastas = [
    ORIGINAL / "data",
    ORIGINAL / "data" / "SABS",
    ORIGINAL / "data" / "CHAOST2",
    REPO / "data" / "SABS",
    REPO / "data" / "CHAOST2",
]

for pasta in pastas:
    print(f"\n=== {pasta} ===")

    if not pasta.is_dir():
        print("NÃO EXISTE")
        continue

    itens = sorted(pasta.iterdir())

    print(f"Existe: SIM ({len(itens)} itens)")

    for item in itens[:20]:
        tipo = "DIR" if item.is_dir() else "FILE"
        print(f"  [{tipo}] {item.name}")

In [ ]:
import importlib.util
import shutil

print(
    "synapseclient instalado:",
    importlib.util.find_spec("synapseclient") is not None
)

print(
    "Comando synapse:",
    shutil.which("synapse")
)

In [ ]:
from pathlib import Path
import subprocess
import sys

# Ambiente separado para downloads
SYNAPSE_ENV = Path.home() / "cassio_pereira" / "synapse-download-env"
SYNAPSE_PYTHON = SYNAPSE_ENV / "bin" / "python"

# Criar ambiente, se necessário
if not SYNAPSE_PYTHON.exists():
    subprocess.run(
        [sys.executable, "-m", "venv", str(SYNAPSE_ENV)],
        check=True
    )

# Instalar cliente oficial
subprocess.run(
    [str(SYNAPSE_PYTHON), "-m", "pip", "install", "synapseclient"],
    check=True
)

# Verificar instalação
subprocess.run(
    [
        str(SYNAPSE_PYTHON),
        "-c",
        "import synapseclient; print('Synapse instalado com sucesso')"
    ],
    check=True
)

In [ ]:
import json
from urllib.request import urlopen

BASE = (
    "https://huggingface.co/api/datasets/"
    "Live12/btcv/tree/main/RawData/Training"
)

for tipo in ["img", "label"]:
    url = f"{BASE}/{tipo}"

    with urlopen(url, timeout=30) as response:
        arquivos = json.load(response)

    nifti = [
        item["path"]
        for item in arquivos
        if item["path"].endswith(".nii.gz")
    ]

    print(f"\n=== {tipo.upper()} ===")
    print("Quantidade:", len(nifti))

    for nome in sorted(nifti)[:5]:
        print(nome)

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import gzip
import shutil

BASE_URL = (
    "https://huggingface.co/datasets/"
    "Live12/btcv/resolve/main/RawData/Training"
)

DESTINO = REPO / "data" / "miccai2015" / "RawData" / "Training"

arquivos = [
    ("img", "img0001.nii.gz"),
    ("label", "label0001.nii.gz"),
]

for pasta, nome in arquivos:
    destino = DESTINO / pasta / nome
    destino.parent.mkdir(parents=True, exist_ok=True)

    if not destino.exists():
        url = f"{BASE_URL}/{pasta}/{nome}"
        temporario = destino.with_name(nome + ".part")

        try:
            print(f"Baixando {nome}...")

            with urlopen(url, timeout=120) as resposta:
                with open(temporario, "wb") as saida:
                    shutil.copyfileobj(resposta, saida)

            temporario.replace(destino)

        finally:
            temporario.unlink(missing_ok=True)

    # Confirmar que o arquivo é um gzip legível
    with gzip.open(destino, "rb") as arquivo:
        cabecalho = arquivo.read(348)

    assert len(cabecalho) == 348, "Cabeçalho incompleto"

    tamanho = destino.stat().st_size / 1024**2

    print(f"OK: {nome} ({tamanho:.2f} MB)")

print("\nArquivos de teste baixados.")

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt

# Arquivos do paciente 0001
IMG = DESTINO / "img" / "img0001.nii.gz"
SEG = DESTINO / "label" / "label0001.nii.gz"

# Carregar os volumes
img_obj = sitk.ReadImage(str(IMG))
seg_obj = sitk.ReadImage(str(SEG))

img = sitk.GetArrayFromImage(img_obj)
seg = sitk.GetArrayFromImage(seg_obj)

# Informações básicas
print("=== VOLUME CT ===")
print("Dimensões (Z, Y, X):", img.shape)
print("Espaçamento (X, Y, Z):", img_obj.GetSpacing())
print("Intensidade mínima:", img.min())
print("Intensidade máxima:", img.max())

print("\n=== MÁSCARA ===")
print("Dimensões:", seg.shape)
print("Classes encontradas:", np.unique(seg))

assert img.shape == seg.shape, "Dimensões incompatíveis!"

# Selecionar a fatia com mais pixels anotados
pixels_por_fatia = np.count_nonzero(seg, axis=(1, 2))
fatia = int(np.argmax(pixels_por_fatia))

print("\nFatia selecionada:", fatia)
print("Pixels anotados:", pixels_por_fatia[fatia])

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(
    img[fatia],
    cmap="gray",
    vmin=-150,
    vmax=250
)
axes[0].set_title("CT original")

axes[1].imshow(seg[fatia], cmap="tab20", interpolation="nearest")
axes[1].set_title("Máscara anotada")

axes[2].imshow(
    img[fatia],
    cmap="gray",
    vmin=-150,
    vmax=250
)

mascara_visivel = np.ma.masked_where(
    seg[fatia] == 0,
    seg[fatia]
)

axes[2].imshow(
    mascara_visivel,
    cmap="tab20",
    alpha=0.5,
    interpolation="nearest"
)
axes[2].set_title("CT + Máscara")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import SimpleITK as sitk
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display

# Carregar CT e máscara
img_obj = sitk.ReadImage(str(DESTINO / "img/img0001.nii.gz"))
seg_obj = sitk.ReadImage(str(DESTINO / "label/label0001.nii.gz"))

img = sitk.GetArrayFromImage(img_obj)
seg = sitk.GetArrayFromImage(seg_obj)

print("Dimensões CT:", img.shape)
print("Dimensões máscara:", seg.shape)
print("Classes:", np.unique(seg))
print("Espaçamento:", img_obj.GetSpacing())

assert img.shape == seg.shape

# Escolher a fatia com mais pixels anotados
z = int(np.argmax(np.count_nonzero(seg, axis=(1, 2))))

ct = img[z]
mask = seg[z]

print("Fatia selecionada:", z)

# Janela CT apenas para visualização
gray = np.uint8(
    np.clip((ct.astype(np.float32) + 150) / 400, 0, 1) * 255
)

rgb = np.stack([gray] * 3, axis=-1)

# Cores para as classes anatômicas
cores = np.array([
    [0, 0, 0],
    [255, 80, 80],
    [80, 220, 100],
    [80, 150, 255],
    [255, 210, 60],
    [210, 80, 255],
    [255, 140, 60],
    [60, 220, 220],
    [230, 100, 180],
    [180, 210, 80],
    [130, 130, 255],
    [255, 180, 130],
    [130, 255, 180],
    [200, 160, 255]
], dtype=np.uint8)

mask_rgb = cores[mask.astype(int) % len(cores)]

overlay = rgb.copy()
fg = mask > 0

overlay[fg] = (
    0.55 * rgb[fg] + 0.45 * mask_rgb[fg]
).astype(np.uint8)

# Montar figura sem Matplotlib
h, w = gray.shape
painel = Image.new("RGB", (w * 3, h + 35), "white")

for i, (titulo, array) in enumerate([
    ("CT ORIGINAL", rgb),
    ("MASCARA", mask_rgb),
    ("CT + MASCARA", overlay)
]):
    painel.paste(Image.fromarray(array), (i * w, 35))
    ImageDraw.Draw(painel).text(
        (i * w + 10, 10),
        titulo,
        fill="black"
    )

display(painel)

In [ ]:
from skimage.segmentation import felzenszwalb, find_boundaries
from PIL import Image
from IPython.display import display
import numpy as np

# Reutilizar a fatia escolhida anteriormente
ct_original = img[z].astype(np.float32)

# Normalizar intensidades para visualização
ct_normalizado = np.clip(
    (ct_original + 150) / 400,
    0, 1
)

# Redimensionar para 256x256
ct_256 = np.asarray(
    Image.fromarray(
        (ct_normalizado * 255).astype(np.uint8)
    ).resize((256, 256), Image.Resampling.BILINEAR),
    dtype=np.float32
) / 255.0

# Felzenszwalb
superpixels = felzenszwalb(
    ct_256,
    scale=1,
    sigma=1,
    min_size=400,
    channel_axis=None
)

# Estatísticas
ids, tamanhos = np.unique(
    superpixels,
    return_counts=True
)

print("=== FELZENSZWALB ===")
print("Resolução:", ct_256.shape)
print("Número de regiões:", len(ids))
print("Menor região:", tamanhos.min(), "pixels")
print("Maior região:", tamanhos.max(), "pixels")
print("Mediana:", np.median(tamanhos), "pixels")

# Criar imagem com fronteiras amarelas
cinza = (ct_256 * 255).astype(np.uint8)
rgb = np.stack([cinza] * 3, axis=-1)

fronteiras = find_boundaries(
    superpixels,
    mode="thick"
)

rgb[fronteiras] = [255, 220, 0]

# Exibir lado a lado
painel = Image.new("RGB", (512, 256))
painel.paste(
    Image.fromarray(cinza).convert("RGB"),
    (0, 0)
)
painel.paste(
    Image.fromarray(rgb),
    (256, 0)
)

display(painel)

In [ ]:
import json

notebook_path = REPO / "data" / "data_processing.ipynb"

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

celula = notebook["cells"][23]
codigo = "".join(celula["source"])

print("=== IMPLEMENTAÇÃO ORIGINAL — CÉLULA 23 ===")

for numero, linha in enumerate(codigo.splitlines(), 1):
    print(f"{numero:03}: {linha}")

In [ ]:
from scipy import ndimage as ndi
from skimage.measure import label
from PIL import Image
from IPython.display import display
import numpy as np

# 1. Máscara do paciente: lógica original
mask_map = ct_256 > 1e-4

componentes = label(mask_map)

if componentes.max() > 0:
    contagens = np.bincount(componentes.ravel())
    contagens[0] = 0

    maior = np.argmax(contagens)
    fg_mask = ndi.binary_fill_holes(componentes == maior)
else:
    fg_mask = mask_map

# 2. Reproduzir superpix_masking SEM correções
raw = superpixels.astype(np.int32).copy()
lbvs = list(np.unique(raw))
max_lb = max(lbvs)

raw[raw == 0] = max_lb + 1
lbvs.append(max_lb)

raw = raw * fg_mask.astype(np.int32)

processed = np.zeros(raw.shape, dtype=np.int32)
lb_new = 1

for lbv in lbvs:
    if lbv == 0:
        continue

    processed[raw == lbv] = lb_new
    lb_new += 1

# 3. Identificar pixels descartados dentro do paciente
perdidos = fg_mask & (processed == 0)

print("=== DIAGNÓSTICO ===")
print("Regiões Felzenszwalb:", len(np.unique(superpixels)))
print("Regiões após processamento:", len(np.unique(processed)) - 1)
print("Pixels na máscara do paciente:", int(fg_mask.sum()))
print("Pixels descartados dentro do paciente:", int(perdidos.sum()))
print("Pixels da região original 0 dentro do paciente:",
      int(((superpixels == 0) & fg_mask).sum()))

# 4. Visualização
gray = (ct_256 * 255).astype(np.uint8)
base = np.stack([gray] * 3, axis=-1)

mascara_rgb = np.stack(
    [(fg_mask * 255).astype(np.uint8)] * 3,
    axis=-1
)

overlay = base.copy()

# Verde: regiões preservadas
overlay[processed > 0] = (
    0.5 * base[processed > 0] +
    0.5 * np.array([0, 255, 0])
).astype(np.uint8)

# Vermelho: pixels descartados dentro do paciente
overlay[perdidos] = [255, 50, 50]

painel = Image.new("RGB", (256 * 3, 256))

for i, array in enumerate([base, mascara_rgb, overlay]):
    painel.paste(Image.fromarray(array), (i * 256, 0))

print("\nOriginal | Máscara do paciente | Verde=preservado, vermelho=descartado")
display(painel)

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil
import gzip

BASE_URL = (
    "https://huggingface.co/datasets/"
    "Live12/btcv/resolve/main/RawData/Training"
)

DESTINO = REPO / "data" / "miccai2015" / "RawData" / "Training"

arquivos = [
    (tipo, f"{prefixo}{i:04d}.nii.gz")
    for i in range(1, 31)
    for tipo, prefixo in [
        ("img", "img"),
        ("label", "label")
    ]
]

baixados = 0
existentes = 0

for indice, (tipo, nome) in enumerate(arquivos, 1):
    destino = DESTINO / tipo / nome
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.exists():
        print(f"[{indice}/60] Já existe: {nome}")
        existentes += 1
        continue

    url = f"{BASE_URL}/{tipo}/{nome}"
    temporario = destino.with_name(nome + ".part")

    print(f"[{indice}/60] Baixando {nome}...", flush=True)

    try:
        with urlopen(url, timeout=120) as resposta:
            with open(temporario, "wb") as saida:
                shutil.copyfileobj(resposta, saida)

        with gzip.open(temporario, "rb") as arquivo:
            assert len(arquivo.read(348)) == 348

        temporario.replace(destino)
        baixados += 1

    finally:
        temporario.unlink(missing_ok=True)

print("\n=== RESULTADO ===")
print("Arquivos baixados:", baixados)
print("Arquivos já existentes:", existentes)
print("Total esperado:", len(arquivos))

print("\n=== ARQUIVOS DISPONÍVEIS ===")
for tipo in ["img", "label"]:
    encontrados = list((DESTINO / tipo).glob("*.nii.gz"))
    print(f"{tipo}: {len(encontrados)}")

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

API = (
    "https://huggingface.co/api/datasets/"
    "Live12/btcv/tree/main/RawData/Training"
)

arquivos_remotos = {}

for tipo in ["img", "label"]:
    with urlopen(f"{API}/{tipo}", timeout=30) as resposta:
        dados = json.load(resposta)

    nomes = sorted(
        Path(item["path"]).name
        for item in dados
        if item["path"].endswith(".nii.gz")
    )

    arquivos_remotos[tipo] = nomes

    print(f"\n=== {tipo.upper()} ===")
    print("Total no Hugging Face:", len(nomes))
    print("Arquivos:", nomes)

    locais = {
        p.name
        for p in (DESTINO / tipo).glob("*.nii.gz")
    }

    faltantes = sorted(set(nomes) - locais)

    print("Já baixados:", len(set(nomes) & locais))
    print("Ainda faltam:", len(faltantes))

# Verificar correspondência entre volumes e máscaras
ids_img = {
    nome.replace("img", "").replace(".nii.gz", "")
    for nome in arquivos_remotos["img"]
}

ids_label = {
    nome.replace("label", "").replace(".nii.gz", "")
    for nome in arquivos_remotos["label"]
}

print("\n=== CORRESPONDÊNCIA ===")
print("IDs em comum:", len(ids_img & ids_label))
print("Imagens sem máscara:", sorted(ids_img - ids_label))
print("Máscaras sem imagem:", sorted(ids_label - ids_img))

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil
import gzip

BASE_URL = (
    "https://huggingface.co/datasets/"
    "Live12/btcv/resolve/main/RawData/Training"
)

baixados = 0
existentes = 0
erros = []

for tipo in ["img", "label"]:
    nomes = arquivos_remotos[tipo]

    for indice, nome in enumerate(nomes, 1):
        destino = DESTINO / tipo / nome
        destino.parent.mkdir(parents=True, exist_ok=True)

        if destino.exists():
            existentes += 1
            continue

        url = f"{BASE_URL}/{tipo}/{nome}"
        temporario = destino.with_name(nome + ".part")

        print(
            f"[{tipo} {indice}/{len(nomes)}] "
            f"Baixando {nome}...",
            flush=True
        )

        try:
            with urlopen(url, timeout=180) as resposta:
                with open(temporario, "wb") as saida:
                    shutil.copyfileobj(resposta, saida)

            # Verificar cabeçalho comprimido
            with gzip.open(temporario, "rb") as arquivo:
                assert len(arquivo.read(348)) == 348

            temporario.replace(destino)
            baixados += 1

        except Exception as erro:
            print(f"ERRO: {nome} — {erro}")
            erros.append(nome)

        finally:
            temporario.unlink(missing_ok=True)

print("\n=== RESULTADO FINAL ===")
print("Novos arquivos baixados:", baixados)
print("Arquivos já existentes:", existentes)
print("Falhas:", erros)

for tipo in ["img", "label"]:
    arquivos = list((DESTINO / tipo).glob("*.nii.gz"))
    print(f"{tipo}: {len(arquivos)}/30 arquivos")

print("\nDownload completo:", (
    len(list((DESTINO / "img").glob("*.nii.gz"))) == 30
    and
    len(list((DESTINO / "label").glob("*.nii.gz"))) == 30
    and not erros
))

In [ ]:
import SimpleITK as sitk
import numpy as np
import gc

PASTA_CT = REPO / "data/miccai2015/RawData/Training"

imagens = sorted((PASTA_CT / "img").glob("*.nii.gz"))

problemas = []
total_fatias = 0
classes_encontradas = set()

print("=== VERIFICAÇÃO DO DATASET CT ===\n")

for img_path in imagens:
    label_path = (
        PASTA_CT / "label" /
        img_path.name.replace("img", "label", 1)
    )

    paciente = img_path.name.replace("img", "").replace(".nii.gz", "")

    try:
        assert label_path.exists(), "Máscara não encontrada"

        imagem = sitk.ReadImage(str(img_path))
        mascara = sitk.ReadImage(str(label_path))

        # Dimensões
        assert imagem.GetSize() == mascara.GetSize(), (
            "Dimensões diferentes"
        )

        # Geometria física
        geometria_ok = all([
            np.allclose(imagem.GetSpacing(), mascara.GetSpacing()),
            np.allclose(imagem.GetOrigin(), mascara.GetOrigin()),
            np.allclose(imagem.GetDirection(), mascara.GetDirection())
        ])

        assert geometria_ok, "Geometria diferente"

        # Classes presentes
        labels = sitk.GetArrayViewFromImage(mascara)
        classes = np.unique(labels).tolist()

        classes_encontradas.update(classes)
        total_fatias += imagem.GetSize()[2]

        assert set(classes).issubset(set(range(14))), (
            f"Classes inesperadas: {classes}"
        )

        print(
            f"Paciente {paciente}: OK | "
            f"Dimensões {imagem.GetSize()} | "
            f"Classes {len(classes)}"
        )

        del labels, imagem, mascara
        gc.collect()

    except Exception as erro:
        problemas.append((paciente, str(erro)))
        print(f"Paciente {paciente}: ERRO — {erro}")

print("\n=== RESUMO ===")
print("Pacientes encontrados:", len(imagens))
print("Fatias totais:", total_fatias)
print("Classes encontradas:", sorted(classes_encontradas))
print("Problemas:", problemas)
print("Integridade OK:", len(imagens) == 30 and len(problemas) == 0)

In [ ]:
import json

notebook_path = REPO / "data" / "data_processing.ipynb"

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

# Células anteriores à geração dos volumes normalizados
indices = [6, 7, 8, 10]

for i in indices:
    celula = notebook["cells"][i]
    codigo = "".join(celula.get("source", []))

    print(f"\n{'=' * 60}")
    print(f"CÉLULA ORIGINAL {i}")
    print(f"{'=' * 60}")

    for numero, linha in enumerate(codigo.splitlines(), 1):
        print(f"{numero:03}: {linha}")

In [ ]:
import json

with open(REPO / "data/data_processing.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

funcoes = [
    "def resample_imgs",
    "def copy_spacing_ori",
    "def resample_by_res",
]

for indice, celula in enumerate(notebook["cells"]):
    codigo = "".join(celula.get("source", []))

    encontradas = [
        nome for nome in funcoes
        if nome in codigo
    ]

    if encontradas:
        print(f"\n{'=' * 60}")
        print(f"CÉLULA ORIGINAL {indice}")
        print("Funções:", encontradas)
        print(f"{'=' * 60}")

        for numero, linha in enumerate(codigo.splitlines(), 1):
            print(f"{numero:03}: {linha}")

In [ ]:
import SimpleITK as sitk
import numpy as np

# Primeiro paciente
caminho = (
    REPO / "data/miccai2015/RawData/Training/img/img0001.nii.gz"
)

imagem = sitk.ReadImage(str(caminho))
array = sitk.GetArrayFromImage(imagem).astype(np.float32)

print("=== IMAGEM ORIGINAL ===")
print("Dimensões:", array.shape)
print("Intensidades:", float(array.min()), float(array.max()))

# Etapa 1: normalização original
LIR = -125
HIR = 275

array = np.clip(array, LIR, HIR)
array = (array - array.min()) / (array.max() - array.min()) * 255.0

print("\n=== APÓS NORMALIZAÇÃO ===")
print("Intensidades:", float(array.min()), float(array.max()))

# Etapa 2: recorte original
BD_BIAS = 32
recortado = array[
    :,
    BD_BIAS:-BD_BIAS,
    BD_BIAS:-BD_BIAS
]

print("\n=== APÓS RECORTE ===")
print("Dimensões:", recortado.shape)

# Etapa 3: redimensionamento original
img_crop = sitk.GetImageFromArray(recortado)
img_crop.SetSpacing(imagem.GetSpacing())
img_crop.SetOrigin(imagem.GetOrigin())
img_crop.SetDirection(imagem.GetDirection())

resolucao = 256
spa_fac = (512 - 2 * BD_BIAS) / resolucao

spacing_original = np.array(imagem.GetSpacing())
spacing_novo = [
    spacing_original[0] * spa_fac,
    spacing_original[1] * spa_fac,
    spacing_original[2]
]

resampler = sitk.ResampleImageFilter()
resampler.SetInterpolator(sitk.sitkLinear)
resampler.SetOutputDirection(img_crop.GetDirection())
resampler.SetOutputOrigin(img_crop.GetOrigin())
resampler.SetOutputSpacing(spacing_novo)

novo_tamanho = (
    np.array(img_crop.GetSize())
    * spacing_original
    / np.array(spacing_novo)
)

resampler.SetSize([int(sz + 1) for sz in novo_tamanho])

resultado = resampler.Execute(img_crop)

print("\n=== RESULTADO FINAL ===")
print("Dimensões (X,Y,Z):", resultado.GetSize())
print("Spacing:", resultado.GetSpacing())
print("Intensidades:", (
    float(sitk.GetArrayViewFromImage(resultado).min()),
    float(sitk.GetArrayViewFromImage(resultado).max())
))

print("\nResolução esperada:", (256, 256))
print("Resolução obtida:", resultado.GetSize()[:2])

In [ ]:
import SimpleITK as sitk
import numpy as np

# Resultado original da célula 21
original = sitk.GetArrayFromImage(resultado)

# Criar redimensionamento sem o +1
resampler_corrigido = sitk.ResampleImageFilter()

resampler_corrigido.SetInterpolator(sitk.sitkLinear)
resampler_corrigido.SetOutputDirection(img_crop.GetDirection())
resampler_corrigido.SetOutputOrigin(img_crop.GetOrigin())
resampler_corrigido.SetOutputSpacing(spacing_novo)

# Preservar as 147 fatias reais
resampler_corrigido.SetSize([256, 256, 147])

corrigido_obj = resampler_corrigido.Execute(img_crop)
corrigido = sitk.GetArrayFromImage(corrigido_obj)

print("=== DIMENSÕES ===")
print("Original:", original.shape)
print("Corrigido:", corrigido.shape)

# Examinar a fatia extra
fatia_extra = original[-1]

print("\n=== FATIA ADICIONAL ===")
print("Mínimo:", float(fatia_extra.min()))
print("Máximo:", float(fatia_extra.max()))
print("Pixels diferentes de zero:",
      int(np.count_nonzero(fatia_extra)))

# Examinar bordas adicionais
ultima_coluna = original[:, :, -1]
ultima_linha = original[:, -1, :]

print("\n=== BORDAS ADICIONAIS ===")
print("Pixels não zero na última coluna:",
      int(np.count_nonzero(ultima_coluna)))
print("Pixels não zero na última linha:",
      int(np.count_nonzero(ultima_linha)))

# Comparar apenas a região correspondente
regiao_original = original[:147, :256, :256]

diferenca = np.abs(regiao_original - corrigido)

print("\n=== COMPARAÇÃO ===")
print("Diferença máxima:", float(diferenca.max()))
print("Diferença média:", float(diferenca.mean()))
print("Pixels idênticos:", bool(np.array_equal(
    regiao_original, corrigido
)))

In [ ]:
import SimpleITK as sitk
import numpy as np
from PIL import Image
from IPython.display import display

# Carregar máscara original
caminho_mask = (
    REPO / "data/miccai2015/RawData/Training/label/label0001.nii.gz"
)

mask_obj = sitk.ReadImage(str(caminho_mask))
mask_original = sitk.GetArrayFromImage(mask_obj)

# Mesmo recorte aplicado à imagem
mask_crop = mask_original[:, 32:-32, 32:-32]

mask_crop_obj = sitk.GetImageFromArray(mask_crop)
mask_crop_obj.SetSpacing(mask_obj.GetSpacing())
mask_crop_obj.SetOrigin(mask_obj.GetOrigin())
mask_crop_obj.SetDirection(mask_obj.GetDirection())

# Reproduzir a interpolação por classe do ProtoSAM
mascara_256 = np.zeros((147, 256, 256), dtype=np.uint8)

classes = np.unique(mask_crop)

for classe in classes:
    if classe == 0:
        continue

    binaria = (mask_crop == classe).astype(np.float32)

    objeto = sitk.GetImageFromArray(binaria)
    objeto.CopyInformation(mask_crop_obj)

    resampler = sitk.ResampleImageFilter()
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetOutputDirection(objeto.GetDirection())
    resampler.SetOutputOrigin(objeto.GetOrigin())
    resampler.SetOutputSpacing(spacing_novo)
    resampler.SetSize([256, 256, 147])

    resultado_classe = sitk.GetArrayFromImage(
        resampler.Execute(objeto)
    )

    mascara_256[np.rint(resultado_classe) == 1] = classe

# Conferir resultados
print("=== MÁSCARA PROCESSADA ===")
print("Original:", mask_original.shape)
print("Após recorte:", mask_crop.shape)
print("Após redimensionamento:", mascara_256.shape)
print("Classes originais:", np.unique(mask_original))
print("Classes finais:", np.unique(mascara_256))

assert mascara_256.shape == corrigido.shape

# Escolher fatia com mais pixels anotados
z_vis = int(np.argmax(
    np.count_nonzero(mascara_256, axis=(1, 2))
))

cinza = np.clip(corrigido[z_vis], 0, 255).astype(np.uint8)
rgb = np.stack([cinza] * 3, axis=-1)

# Cores para visualização
cores = np.random.default_rng(42).integers(
    60, 255, size=(14, 3), dtype=np.uint8
)
cores[0] = 0

overlay = rgb.copy()
foreground = mascara_256[z_vis] > 0

overlay[foreground] = (
    0.5 * rgb[foreground]
    + 0.5 * cores[mascara_256[z_vis][foreground]]
).astype(np.uint8)

painel = Image.new("RGB", (512, 256))
painel.paste(Image.fromarray(rgb), (0, 0))
painel.paste(Image.fromarray(overlay), (256, 0))

print("\nFatia:", z_vis)
print("Esquerda: CT normalizado | Direita: máscara sobreposta")

display(painel)

In [ ]:
from skimage.segmentation import felzenszwalb, find_boundaries
from skimage.measure import label
from scipy.ndimage import binary_fill_holes
from PIL import Image
from IPython.display import display
import numpy as np

# Imagem CT já normalizada e redimensionada
fatia_ct = corrigido[z_vis]

# 1. Felzenszwalb original
regioes = felzenszwalb(
    fatia_ct,
    scale=1,
    sigma=1,
    min_size=400,
    channel_axis=None
)

# 2. Máscara do paciente (lógica original)
mask_map = fatia_ct > 1e-4
componentes = label(mask_map)

if componentes.max() > 0:
    contagens = np.bincount(componentes.ravel())
    contagens[0] = 0
    maior = np.argmax(contagens)

    fg_mask = binary_fill_holes(componentes == maior)
else:
    fg_mask = mask_map

# 3. Aplicar superpix_masking original
raw = regioes.astype(np.int32).copy()
lbvs = list(np.unique(raw))
max_lb = max(lbvs)

raw[raw == 0] = max_lb + 1
lbvs.append(max_lb)
raw = raw * fg_mask.astype(np.int32)

processadas = np.zeros(raw.shape, dtype=np.int32)
lb_new = 1

for lbv in lbvs:
    if lbv == 0:
        continue

    processadas[raw == lbv] = lb_new
    lb_new += 1

# 4. Estatísticas
ids, tamanhos = np.unique(
    processadas[processadas > 0],
    return_counts=True
)

perdidos = fg_mask & (processadas == 0)

print("=== RESULTADOS ===")
print("Fatia:", z_vis)
print("Regiões originais:", len(np.unique(regioes)))
print("Regiões válidas:", len(ids))
print("Pixels do paciente:", int(fg_mask.sum()))
print("Pixels descartados dentro do paciente:", int(perdidos.sum()))

if len(tamanhos):
    print("Menor região:", int(tamanhos.min()))
    print("Maior região:", int(tamanhos.max()))
    print("Mediana:", float(np.median(tamanhos)))

# 5. Visualização sem Matplotlib
cinza = np.clip(fatia_ct, 0, 255).astype(np.uint8)
rgb = np.stack([cinza] * 3, axis=-1)

visualizacao = rgb.copy()
fronteiras = find_boundaries(processadas, mode="thick")
visualizacao[fronteiras] = [255, 220, 0]

mascara_paciente = (
    fg_mask.astype(np.uint8) * 255
)

painel = Image.new("RGB", (768, 256))
painel.paste(Image.fromarray(rgb), (0, 0))
painel.paste(Image.fromarray(visualizacao), (256, 0))
painel.paste(
    Image.fromarray(mascara_paciente).convert("RGB"),
    (512, 0)
)

print("\nOriginal | Superpixels | Máscara do paciente")
display(painel)

In [ ]:
import numpy as np
from skimage.segmentation import felzenszwalb
from skimage.measure import label
from scipy.ndimage import binary_fill_holes

diagnostico_ct0001 = []

for z_idx, fatia in enumerate(corrigido):

    # 1. Felzenszwalb original
    regioes = felzenszwalb(
        fatia,
        scale=1,
        sigma=1,
        min_size=400,
        channel_axis=None
    )

    # 2. Máscara do paciente
    mask_map = fatia > 1e-4

    if mask_map.any():
        componentes = label(mask_map)
        contagens = np.bincount(componentes.ravel())
        contagens[0] = 0

        maior = np.argmax(contagens)
        fg = binary_fill_holes(componentes == maior)
    else:
        fg = mask_map

    # 3. Diagnosticar a lógica original de superpix_masking
    # O rótulo original 0 é descartado pela implementação.
    pixels_perdidos = np.count_nonzero(
        fg & (regioes == 0)
    )

    ids_validos = np.unique(regioes[fg])
    ids_validos = ids_validos[ids_validos != 0]

    diagnostico_ct0001.append({
        "fatia": z_idx,
        "regioes": len(ids_validos),
        "pixels_paciente": int(fg.sum()),
        "pixels_perdidos": int(pixels_perdidos)
    })

# Resumo
com_perda = [
    r for r in diagnostico_ct0001
    if r["pixels_perdidos"] > 0
]

com_paciente = [
    r for r in diagnostico_ct0001
    if r["pixels_paciente"] > 0
]

print("=== DIAGNÓSTICO DO PACIENTE 0001 ===")
print("Fatias analisadas:", len(diagnostico_ct0001))
print("Fatias com paciente:", len(com_paciente))
print("Fatias com perda de pixels:", len(com_perda))

if com_paciente:
    quantidades = [r["regioes"] for r in com_paciente]
    print("Regiões por fatia (mínimo):", min(quantidades))
    print("Regiões por fatia (máximo):", max(quantidades))
    print("Regiões por fatia (mediana):", np.median(quantidades))

print("\n=== MAIORES PERDAS ===")
for registro in sorted(
    com_perda,
    key=lambda r: r["pixels_perdidos"],
    reverse=True
)[:10]:
    print(registro)

print("\nTotal de pixels perdidos:",
      sum(r["pixels_perdidos"] for r in com_perda))

In [ ]:
import json
import os
import numpy as np
import SimpleITK as sitk
from pathlib import Path

ORIGEM = REPO / "data/miccai2015/RawData/Training"
SAIDA = REPO / "data/SABS/sabs_CT_normalized"

SAIDA.mkdir(parents=True, exist_ok=True)

arquivos = sorted((ORIGEM / "img").glob("*.nii.gz"))
assert len(arquivos) == 30

def preparar_objeto(array, referencia):
    obj = sitk.GetImageFromArray(array)
    obj.SetSpacing(referencia.GetSpacing())
    obj.SetOrigin(referencia.GetOrigin())
    obj.SetDirection(referencia.GetDirection())
    return obj

def criar_resampler(referencia, z):
    spacing = referencia.GetSpacing()

    r = sitk.ResampleImageFilter()
    r.SetInterpolator(sitk.sitkLinear)
    r.SetOutputOrigin(referencia.GetOrigin())
    r.SetOutputDirection(referencia.GetDirection())
    r.SetOutputSpacing((
        spacing[0] * 448 / 256,
        spacing[1] * 448 / 256,
        spacing[2]
    ))
    r.SetSize([256, 256, z])
    return r

manifesto = []

for indice, caminho_img in enumerate(arquivos):
    paciente = caminho_img.name.replace("img", "").replace(".nii.gz", "")
    caminho_label = ORIGEM / "label" / f"label{paciente}.nii.gz"

    saida_img = SAIDA / f"image_{indice}.nii.gz"
    saida_label = SAIDA / f"label_{indice}.nii.gz"

    manifesto.append({
        "indice": indice,
        "paciente_original": paciente,
        "imagem": saida_img.name,
        "mascara": saida_label.name
    })

    if saida_img.exists() and saida_label.exists():
        print(f"[{indice+1}/30] Paciente {paciente}: já processado")
        continue

    if saida_img.exists() or saida_label.exists():
        raise RuntimeError(
            f"Paciente {paciente}: par de saída incompleto"
        )

    img_obj = sitk.ReadImage(str(caminho_img))
    label_obj = sitk.ReadImage(str(caminho_label))

    img = sitk.GetArrayFromImage(img_obj).astype(np.float32)
    label = sitk.GetArrayFromImage(label_obj)

    assert img.shape == label.shape

    # Normalização CT original
    img = np.clip(img, -125, 275)
    amplitude = img.max() - img.min()
    assert amplitude > 0

    img = (img - img.min()) / amplitude * 255.0

    # Recorte de 32 pixels
    img = img[:, 32:-32, 32:-32]
    label = label[:, 32:-32, 32:-32]

    z = img.shape[0]

    img_crop = preparar_objeto(img, img_obj)
    label_crop = preparar_objeto(label, label_obj)

    # Redimensionamento 256 (sem o pixel adicional)
    r = criar_resampler(img_crop, z)
    img_final = r.Execute(img_crop)

    # Máscara: interpolação por classe do ProtoSAM
    label_final = np.zeros((z, 256, 256), dtype=np.uint8)

    for classe in np.unique(label):
        if classe == 0:
            continue

        binaria = (label == classe).astype(np.float32)
        binaria_obj = preparar_objeto(binaria, label_crop)

        resultado = sitk.GetArrayFromImage(r.Execute(binaria_obj))
        label_final[np.rint(resultado) == 1] = classe

    label_final_obj = sitk.GetImageFromArray(label_final)
    label_final_obj.CopyInformation(img_final)

    # Salvar arquivos processados
    sitk.WriteImage(img_final, str(saida_img), True)
    sitk.WriteImage(label_final_obj, str(saida_label), True)

    print(
        f"[{indice+1}/30] Paciente {paciente}: "
        f"OK | {z} fatias | 256x256",
        flush=True
    )

# Registrar correspondência entre IDs
with open(SAIDA / "manifesto_pacientes.json", "w") as f:
    json.dump({
        "dataset": "SABS",
        "resolucao": 256,
        "normalizacao_hu": [-125, 275],
        "recorte_borda": 32,
        "tamanho_corrigido": True,
        "pacientes": manifesto
    }, f, indent=2)

print("\n=== RESUMO ===")
print("Imagens:", len(list(SAIDA.glob("image_*.nii.gz"))))
print("Máscaras:", len(list(SAIDA.glob("label_*.nii.gz"))))
print("Diretório:", SAIDA)

In [ ]:
import json

notebook_path = REPO / "data" / "data_processing.ipynb"

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

codigo = "".join(
    notebook["cells"][24].get("source", [])
)

print("=== GERAÇÃO ORIGINAL DE PSEUDOMÁSCARAS ===\n")

for numero, linha in enumerate(codigo.splitlines(), 1):
    print(f"{numero:03}: {linha}")

In [ ]:
import SimpleITK as sitk
import numpy as np

from pathlib import Path
from skimage.segmentation import felzenszwalb
from skimage.measure import label
from scipy.ndimage import binary_fill_holes

PASTA = REPO / "data/SABS/sabs_CT_normalized"

MODE = "MIDDLE"
FG_THRESH = 1e-4


def gerar_pseudomascaras(volume):

    fg_volume = np.zeros(volume.shape, dtype=np.float64)
    sp_volume = np.zeros(volume.shape, dtype=np.float64)

    for z in range(volume.shape[0]):

        fatia = volume[z]

        # 1. Felzenszwalb original
        regioes = felzenszwalb(
            fatia,
            scale=1,
            sigma=1,
            min_size=400,
            channel_axis=None
        )

        # 2. Máscara do paciente
        mask_map = (fatia > FG_THRESH).astype(np.float32)

        if mask_map.max() < 0.999:
            fg = mask_map
        else:
            componentes = label(mask_map)

            contagens = np.bincount(componentes.ravel())
            contagens[0] = 0

            maior = np.argmax(contagens)
            fg = binary_fill_holes(componentes == maior)

        # 3. Renumeração original do ProtoSAM
        raw = regioes.astype(np.int32).copy()

        lbvs = list(np.unique(raw))
        max_lb = max(lbvs)

        raw[raw == 0] = max_lb + 1
        lbvs.append(max_lb)

        raw = raw * fg.astype(np.int32)

        processadas = np.zeros(raw.shape)

        lb_new = 1

        for lbv in lbvs:
            if lbv == 0:
                continue

            processadas[raw == lbv] = lb_new
            lb_new += 1

        fg_volume[z] = fg
        sp_volume[z] = processadas

    return fg_volume, sp_volume


arquivos = sorted(
    PASTA.glob("image_*.nii.gz"),
    key=lambda p: int(p.stem.split("_")[-1].split(".")[0])
)

assert len(arquivos) == 30, "Esperados 30 volumes CT"

print("=== GERANDO PSEUDOMÁSCARAS CT ===", flush=True)

concluidos = 0

for indice, caminho in enumerate(arquivos, 1):

    pid = caminho.name.split("_")[-1].split(".")[0]

    saida_sp = PASTA / f"superpix-{MODE}_{pid}.nii.gz"
    saida_fg = PASTA / f"fgmask_{pid}.nii.gz"

    if saida_sp.exists() and saida_fg.exists():
        print(f"[{indice}/30] Volume {pid}: já existe", flush=True)
        concluidos += 1
        continue

    if saida_sp.exists() or saida_fg.exists():
        raise RuntimeError(
            f"Volume {pid}: apenas um dos arquivos existe. "
            "Verifique antes de continuar."
        )

    print(f"[{indice}/30] Processando volume {pid}...", flush=True)

    imagem = sitk.ReadImage(str(caminho))
    volume = sitk.GetArrayFromImage(imagem)

    fg, sp = gerar_pseudomascaras(volume)

    assert fg.shape == volume.shape
    assert sp.shape == volume.shape

    fg_obj = sitk.GetImageFromArray(fg)
    sp_obj = sitk.GetImageFromArray(sp)

    fg_obj.CopyInformation(imagem)
    sp_obj.CopyInformation(imagem)

    # Arquivos temporários para evitar saídas incompletas
    temp_fg = PASTA / f"fgmask_{pid}.tmp.nii.gz"
    temp_sp = PASTA / f"superpix-{MODE}_{pid}.tmp.nii.gz"

    sitk.WriteImage(fg_obj, str(temp_fg), True)
    sitk.WriteImage(sp_obj, str(temp_sp), True)

    temp_fg.replace(saida_fg)
    temp_sp.replace(saida_sp)

    concluidos += 1

    print(
        f"  OK | Fatias: {volume.shape[0]} | "
        f"Regiões na primeira fatia: "
        f"{len(np.unique(sp[0][sp[0] > 0]))}",
        flush=True
    )

print("\n=== RESULTADO ===")
print("Volumes concluídos:", concluidos)
print("Pseudomáscaras:", len(list(PASTA.glob("superpix-MIDDLE_*.nii.gz"))))
print("Máscaras de paciente:", len(list(PASTA.glob("fgmask_*.nii.gz"))))

In [ ]:
import re

arquivos = [
    REPO / "backbone.sh",
    REPO / "config_ssl_upload.py",
    REPO / "training.py",
    REPO / "dataloaders/GenericSuperDatasetv2.py",
    REPO / "dataloaders/ManualAnnoDatasetv2.py",
]

padrao = re.compile(
    r"SABS|LABEL_SET|LABEL_GROUP|exclude|excl_"
    r"|superpix-MIDDLE|fgmask|label_sets|label_group",
    re.IGNORECASE
)

for arquivo in arquivos:
    if not arquivo.exists():
        continue

    linhas = arquivo.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    indices = set()

    for i, linha in enumerate(linhas):
        if padrao.search(linha):
            indices.update(
                range(max(0, i - 3), min(len(linhas), i + 5))
            )

    print(f"\n=== {arquivo.relative_to(REPO)} ===")

    if not indices:
        print("Nenhuma referência encontrada")
        continue

    for i in sorted(indices)[:90]:
        print(f"{i+1:04}: {linhas[i]}")

    if len(indices) > 90:
        print(f"... {len(indices) - 90} linhas adicionais omitidas")

In [ ]:
import json
from pathlib import Path

PASTA_CT = REPO / "data/SABS/sabs_CT_normalized"

print("=== ARQUIVOS DO EFT ===")

for nome in [
    "image_0.nii.gz",
    "label_0.nii.gz",
    "superpix-MIDDLE_0.nii.gz",
    "fgmask_0.nii.gz",
    "classmap_1.json",
]:
    caminho = PASTA_CT / nome
    print(f"{nome}: {'OK' if caminho.exists() else 'AUSENTE'}")

print("\n=== GERAÇÃO ORIGINAL DO CLASSMAP ===")

with open(REPO / "data/data_processing.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

codigo = "".join(notebook["cells"][12]["source"])
linhas = codigo.splitlines()

for i, linha in enumerate(linhas):
    if "classmap = {}" in linha:
        for numero in range(i, len(linhas)):
            print(f"{numero+1:03}: {linhas[numero]}")
        break
else:
    print("Trecho de geração do classmap não encontrado.")

In [ ]:
import json
import numpy as np
import SimpleITK as sitk

PASTA_CT = REPO / "data/SABS/sabs_CT_normalized"

LABEL_NAME = [
    "BGD", "SPLEEN", "KID_R", "KID_l",
    "GALLBLADDER", "ESOPHAGUS", "LIVER",
    "STOMACH", "AORTA", "IVC",
    "PS_VEIN", "PANCREAS", "AG_R", "AG_L"
]

SAIDA = PASTA_CT / "classmap_1.json"

# Evitar sobrescrever o arquivo
if SAIDA.exists():
    print("Classmap já existe:", SAIDA)

else:
    arquivos = sorted(
        PASTA_CT.glob("label_*.nii.gz"),
        key=lambda p: int(p.name.split("_")[1].split(".")[0])
    )

    assert len(arquivos) == 30

    classmap = {
        nome: {} for nome in LABEL_NAME
    }

    total_fatias = 0

    for caminho in arquivos:
        pid = caminho.name.split("_")[1].split(".")[0]

        volume = sitk.GetArrayFromImage(
            sitk.ReadImage(str(caminho))
        )

        for nome in LABEL_NAME:
            classmap[nome][pid] = []

        for z, fatia in enumerate(volume):
            classes = np.unique(fatia)

            for classe in classes:
                classe = int(classe)
                assert 0 <= classe < len(LABEL_NAME)
                classmap[LABEL_NAME[classe]][pid].append(z)

        total_fatias += len(volume)
        print(f"Paciente {pid}: OK")

    with open(SAIDA, "w", encoding="utf-8") as f:
        json.dump(classmap, f, indent=2)

    print("\n=== RESULTADO ===")
    print("Pacientes:", len(arquivos))
    print("Fatias:", total_fatias)

    for nome in ["SPLEEN", "KID_R", "KID_l", "LIVER"]:
        quantidade = sum(
            len(fatias)
            for fatias in classmap[nome].values()
        )
        print(f"{nome}: {quantidade} fatias")

    print("\nClassmap salvo:", SAIDA)

In [ ]:
import json
import SimpleITK as sitk

PASTA = REPO / "data/SABS/sabs_CT_normalized"

with open(PASTA / "classmap_1.json") as f:
    classmap = json.load(f)

total = 0
excluidas = 0
elegiveis = 0

print("=== EXCLUSÃO DE CLASSES — CT ===")

for pid in range(30):
    pid = str(pid)

    imagem = sitk.ReadImage(
        str(PASTA / f"image_{pid}.nii.gz")
    )

    n_fatias = imagem.GetSize()[2]

    # Classes 2 e 3: rim direito e esquerdo
    fatias_excluidas = (
        set(classmap["KID_R"][pid])
        | set(classmap["KID_l"][pid])
    )

    assert all(
        0 <= z < n_fatias
        for z in fatias_excluidas
    )

    total += n_fatias
    excluidas += len(fatias_excluidas)
    elegiveis += n_fatias - len(fatias_excluidas)

print("Total de fatias:", total)
print("Fatias contendo rins:", excluidas)
print("Fatias elegíveis:", elegiveis)
print(
    "Percentual elegível:",
    f"{100 * elegiveis / total:.2f}%"
)

print("\n=== IMPLEMENTAÇÃO DO DATALOADER ===")

caminho = REPO / "dataloaders/GenericSuperDatasetv2.py"
linhas = caminho.read_text().splitlines()

for i, linha in enumerate(linhas):
    if "for _ex_cls in self.exclude_lbs:" in linha:
        for j in range(max(0, i - 2), min(len(linhas), i + 6)):
            print(f"{j+1}: {linhas[j]}")

In [ ]:
from pathlib import Path
import ast

print("=== 1. DEFINIÇÃO DOS FOLDS ===")

for arquivo in (REPO / "dataloaders").glob("*.py"):
    codigo = arquivo.read_text(
        encoding="utf-8", errors="replace"
    )

    try:
        arvore = ast.parse(codigo)
    except SyntaxError:
        continue

    for no in ast.walk(arvore):
        if isinstance(no, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if no.name == "get_scanids":
                linhas = codigo.splitlines()

                print(f"\nArquivo: {arquivo.relative_to(REPO)}")

                for i in range(
                    no.lineno - 1,
                    min(no.end_lineno, len(linhas))
                ):
                    print(f"{i+1:04}: {linhas[i]}")


print("\n=== 2. SCRIPTS EFT EXISTENTES ===")

pasta_exp = REPO / "exp"

for arquivo in sorted(pasta_exp.rglob("*")):
    if not arquivo.is_file():
        continue

    nome = arquivo.name.lower()

    if (
        "eft" in nome
        or "treinar" in nome
        or "training" in nome
    ):
        print(arquivo.relative_to(REPO))


print("\n=== 3. CONFIGURAÇÃO DO BACKBONE ORIGINAL ===")

arquivo = REPO / "backbone.sh"
linhas = arquivo.read_text().splitlines()

termos = [
    "MODEL", "NSTEP", "BACKBONE",
    "python", "training.py", "BASE_MODEL"
]

for i, linha in enumerate(linhas, 1):
    if any(t.lower() in linha.lower() for t in termos):
        print(f"{i:04}: {linha[:200]}")

In [ ]:
from pathlib import Path
import re

SCRIPT = (
    REPO / "exp/eft_tiny_512_v1/"
    "treinar_SP256_1231_20261021T194500705194Z.py"
)

print("=== SCRIPT DE REFERÊNCIA ===")
print("Existe:", SCRIPT.exists())

if SCRIPT.exists():
    linhas = SCRIPT.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    print("Total de linhas:", len(linhas))

    print("\n=== IMPORTS E CONFIGURAÇÃO INICIAL ===")
    for i, linha in enumerate(linhas[:90], 1):
        print(f"{i:04}: {linha}")

    print("\n=== PONTOS IMPORTANTES DO TREINAMENTO ===")

    padrao = re.compile(
        r"optimizer|AdamW|checkpoint|torch.save|"
        r"DataLoader|manual_seed|learning_rate|"
        r"n_passos|total_passos|num_steps|"
        r"if __name__|def main|requires_grad",
        re.IGNORECASE
    )

    encontrados = [
        (i, linha.strip())
        for i, linha in enumerate(linhas, 1)
        if padrao.search(linha)
    ]

    for numero, linha in encontrados[:45]:
        print(f"{numero:04}: {linha[:180]}")

print("\n=== DIVISÃO CT — FOLD 0 ===")

# Conforme a configuração SABS encontrada anteriormente
ids = list(range(30))
separadores = [0, 6, 12, 18, 24, 30]
n_shots = 1

ids_reservados = ids[
    separadores[0]:separadores[1] + n_shots
]
ids_treino = [
    i for i in ids if i not in ids_reservados
]

print("Pacientes reservados:", ids_reservados)
print("Pacientes de treino:", ids_treino)
print("Total de pacientes para treino:", len(ids_treino))

In [ ]:
from pathlib import Path

SCRIPT = (
    REPO / "exp/eft_tiny_512_v1/"
    "treinar_SP256_1231_20261021T194500705194Z.py"
)

linhas = SCRIPT.read_text(encoding="utf-8").splitlines()

trechos = {
    "1. Construção dos episódios": (65, 139),
    "2. Modelo e otimizador": (225, 245),
    "3. Loop de treinamento": (284, 340),
}

for titulo, (inicio, fim) in trechos.items():
    print(f"\n{'=' * 65}")
    print(titulo)
    print("=" * 65)

    for numero in range(inicio, min(fim, len(linhas)) + 1):
        print(f"{numero:04}: {linhas[numero - 1]}")

In [ ]:
import os
import sys
import gc
import json
import numpy as np
import torch
import torch.nn.functional as F
import SimpleITK as sitk

# Reutilizar implementação validada do VMamba
MODULOS = REPO / "exp/eft_tiny_512_v1"

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(MODULOS))
os.chdir(REPO)

from eft_modelo_672 import construir_modelo
from eft_alinhamento_seguro import instalar

assert torch.cuda.is_available(), "CUDA indisponível"

DADOS_CT = REPO / "data/SABS/sabs_CT_normalized"

with open(DADOS_CT / "classmap_1.json") as f:
    classmap = json.load(f)

# Buscar uma região válida nos pacientes de treino do fold 0
episodio_ct = None

for pid in range(7, 30):
    imagem = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS_CT / f"image_{pid}.nii.gz"))
    )

    regioes = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS_CT / f"superpix-MIDDLE_{pid}.nii.gz"))
    )

    excluidas = (
        set(classmap["KID_R"][str(pid)])
        | set(classmap["KID_l"][str(pid)])
    )

    for z in range(len(imagem)):
        if z in excluidas:
            continue

        ids, areas = np.unique(
            regioes[z][regioes[z] > 0],
            return_counts=True
        )

        candidatas = [
            (int(i), int(a))
            for i, a in zip(ids, areas)
            if 400 <= a <= 10000
        ]

        if candidatas:
            regiao, area = candidatas[0]

            episodio_ct = {
                "paciente": pid,
                "fatia": z,
                "regiao": regiao,
                "area": area,
                "imagem": imagem[z].copy(),
                "mascara": (regioes[z] == regiao)
            }
            break

    if episodio_ct is not None:
        break

assert episodio_ct is not None, "Nenhum episódio válido"

print("=== EPISÓDIO SELECIONADO ===")
print("Paciente:", episodio_ct["paciente"])
print("Fatia:", episodio_ct["fatia"])
print("Região:", episodio_ct["regiao"])
print("Área:", episodio_ct["area"])

# Converter CT grayscale em três canais
rgb = np.repeat(
    episodio_ct["imagem"][..., None],
    3,
    axis=2
).astype(np.float32)

mask = episodio_ct["mascara"]

def preparar_imagem(array):
    tensor = torch.from_numpy(
        np.ascontiguousarray(array)
    ).permute(2, 0, 1)[None].to(
        "cuda", dtype=torch.float32
    )

    media = tensor.new_tensor(
        [123.675, 116.28, 103.53]
    )[None, :, None, None]

    desvio = tensor.new_tensor(
        [58.395, 57.12, 57.375]
    )[None, :, None, None]

    return (tensor - media) / desvio

support = preparar_imagem(rgb)
query = preparar_imagem(np.fliplr(rgb))

fg = torch.from_numpy(
    mask.copy()
)[None].to("cuda", dtype=torch.float32)

alvo = torch.from_numpy(
    np.fliplr(mask).copy()
)[None].to("cuda", dtype=torch.long)

print("\n=== TENSORES ===")
print("Support:", tuple(support.shape))
print("Query:", tuple(query.shape))
print("Máscara:", tuple(fg.shape))

# Construir modelo do zero a partir dos pesos pré-treinados
modelo = construir_modelo("tiny", repo=REPO)
estado_align = instalar(modelo)
modelo.train()
modelo.zero_grad(set_to_none=True)

print("\n=== FORWARD ===", flush=True)

retorno = modelo(
    [[support]],
    [[fg]],
    [[1 - fg]],
    [query],
    isval=False,
    val_wsize=2,
    show_viz=False
)

logits, alinhamento = retorno[:2]

print("Logits:", tuple(logits.shape))
print("Alinhamento:", float(alinhamento))

assert logits.shape == (1, 2, 256, 256)
assert torch.isfinite(logits).all()

ce = F.cross_entropy(logits, alvo)
loss = ce + alinhamento

print("\n=== LOSS ===")
print("Cross entropy:", float(ce))
print("Alinhamento:", float(alinhamento))
print("Total:", float(loss))

assert torch.isfinite(loss)

# Backward sem optimizer.step()
print("\n=== BACKWARD ===", flush=True)

loss.backward()

gradientes = [
    p.grad for p in modelo.encoder.parameters()
    if p.grad is not None
]

nao_nulos = sum(
    int(torch.count_nonzero(g).item() > 0)
    for g in gradientes
)

print("Tensores com gradiente:", len(gradientes))
print("Gradientes não nulos:", nao_nulos)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Memória máxima:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

assert gradientes and nao_nulos > 0

print("\nTESTE EFT-CT: OK — pesos não atualizados")

# Liberar memória da GPU
del modelo, retorno, logits, alinhamento, ce, loss
del support, query, fg, alvo, gradientes, estado_align
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import json
import random
from collections import Counter
import numpy as np
import SimpleITK as sitk

DADOS = REPO / "data/SABS/sabs_CT_normalized"
EXPERIMENTO = REPO / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
EXPERIMENTO.mkdir(parents=True, exist_ok=True)

PLANO = EXPERIMENTO / "plano_200.json"

assert not PLANO.exists(), "Plano já existe. Não será sobrescrito."

with open(DADOS / "classmap_1.json") as f:
    classmap = json.load(f)

candidatos = []
total_fatias_elegiveis = 0

for pid in range(7, 30):

    caminho_img = DADOS / f"image_{pid}.nii.gz"
    caminho_sp = DADOS / f"superpix-MIDDLE_{pid}.nii.gz"

    imagem_obj = sitk.ReadImage(str(caminho_img))
    sp_obj = sitk.ReadImage(str(caminho_sp))

    assert imagem_obj.GetSize() == sp_obj.GetSize()

    regioes = sitk.GetArrayFromImage(sp_obj)

    excluidas = (
        set(classmap["KID_R"][str(pid)])
        | set(classmap["KID_l"][str(pid)])
    )

    for z in range(regioes.shape[0]):

        if z in excluidas:
            continue

        total_fatias_elegiveis += 1

        ids, areas = np.unique(
            regioes[z],
            return_counts=True
        )

        for regiao, area in zip(ids, areas):

            regiao = int(regiao)
            area = int(area)

            # Critério do piloto: evitar regiões
            # muito pequenas ou excessivamente grandes.
            if regiao <= 0 or not (400 <= area <= 10000):
                continue

            candidatos.append({
                "paciente": pid,
                "fatia": z,
                "regiao": regiao,
                "area": area,
            })

assert len(candidatos) >= 200

rng = random.Random(1234)
episodios = rng.sample(candidatos, 200)

# Verificação de exclusão
for ep in episodios:
    pid = str(ep["paciente"])
    z = ep["fatia"]

    assert z not in classmap["KID_R"][pid]
    assert z not in classmap["KID_l"][pid]

plano = {
    "dataset": "SABS",
    "modalidade": "CT",
    "fold": 0,
    "pacientes_treino": list(range(7, 30)),
    "classes_excluidas": [2, 3],
    "metodo": "EFT Felzenszwalb",
    "resolucao": 256,
    "semente": 1234,
    "total_passos": 200,
    "min_area": 400,
    "max_area": 10000,
    "episodios": episodios,
}

temporario = PLANO.with_suffix(".json.tmp")
temporario.write_text(
    json.dumps(plano, indent=2),
    encoding="utf-8"
)
temporario.replace(PLANO)

distribuicao = Counter(
    ep["paciente"] for ep in episodios
)

print("=== PLANO EFT-CT ===")
print("Pacientes de treino:", 23)
print("Fatias elegíveis no fold:", total_fatias_elegiveis)
print("Regiões candidatas:", len(candidatos))
print("Episódios selecionados:", len(episodios))
print("Pacientes representados:", len(distribuicao))
print("Classes excluídas:", [2, 3])
print("Vazamento de classes excluídas: NÃO")
print("\nPlano salvo:", PLANO)
print("\nPrimeiro episódio:", episodios[0])

In [ ]:
from pathlib import Path

PASTA_SCRIPT = REPO / "exp/eft_ct_piloto200_v1"
PASTA_SCRIPT.mkdir(parents=True, exist_ok=True)

SCRIPT_CT = PASTA_SCRIPT / "treinar_ct.py"

codigo = r'''
import os
import sys
import json
import time
import random
import hashlib
import traceback
from pathlib import Path
from functools import lru_cache

import numpy as np
import torch
import torch.nn.functional as F
import SimpleITK as sitk

ROOT = Path(__file__).resolve().parents[2]
MODULOS = ROOT / "exp/eft_tiny_512_v1"

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(MODULOS))

from eft_modelo_672 import construir_modelo
from eft_alinhamento_seguro import instalar

DADOS = ROOT / "data/SABS/sabs_CT_normalized"
RUN = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
PLANO = RUN / "plano_200.json"

SEMENTE = 1234
PASSOS = 200
LR = 1e-5
WD = 0.01

MEDIA = [123.675, 116.28, 103.53]
DESVIO = [58.395, 57.12, 57.375]


def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def salvar_json(caminho, dados):
    temp = caminho.with_suffix(caminho.suffix + ".tmp")
    temp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False)
    )
    os.replace(temp, caminho)


@lru_cache(maxsize=4)
def carregar_volume(pid):
    imagem = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
    )
    regioes = sitk.GetArrayFromImage(
        sitk.ReadImage(str(
            DADOS / f"superpix-MIDDLE_{pid}.nii.gz"
        ))
    )

    assert imagem.shape == regioes.shape
    assert imagem.shape[1:] == (256, 256)

    return imagem, regioes


def preparar_imagem(array):
    t = torch.from_numpy(
        np.ascontiguousarray(array)
    ).permute(2, 0, 1)[None].to(
        "cuda", dtype=torch.float32
    )

    media = t.new_tensor(MEDIA)[None, :, None, None]
    desvio = t.new_tensor(DESVIO)[None, :, None, None]

    return (t - media) / desvio


def episodio(registro):
    pid = int(registro["paciente"])
    z = int(registro["fatia"])
    regiao = int(registro["regiao"])

    imagem, regioes = carregar_volume(pid)

    fatia = imagem[z].astype(np.float32)
    mascara = regioes[z] == regiao

    assert mascara.any()
    assert int(mascara.sum()) == int(registro["area"])

    rgb = np.repeat(fatia[..., None], 3, axis=2)

    support = preparar_imagem(rgb)
    query = preparar_imagem(np.fliplr(rgb))

    fg = torch.from_numpy(
        mascara.copy()
    )[None].to("cuda", dtype=torch.float32)

    alvo = torch.from_numpy(
        np.fliplr(mascara).copy()
    )[None].to("cuda", dtype=torch.long)

    return support, fg, query, alvo


def salvar_checkpoint(passo, modelo, otimizador):
    estado = {
        "passo": passo,
        "modelo": modelo.state_dict(),
        "otimizador": otimizador.state_dict(),
        "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(),
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all(),
    }

    destino = RUN / f"checkpoint_{passo:04d}.pt"
    temp = RUN / f"checkpoint_{passo:04d}.pt.tmp"

    torch.save(estado, temp)
    os.replace(temp, destino)

    print(f"Checkpoint salvo: {passo}", flush=True)


def treinar():
    assert torch.cuda.is_available()

    if (RUN / "checkpoint_0000.pt").exists():
        raise RuntimeError(
            "Já existe um treinamento iniciado. "
            "Não iniciar novamente."
        )

    plano = json.loads(PLANO.read_text())
    episodios = plano["episodios"]

    assert len(episodios) == PASSOS
    assert plano["fold"] == 0
    assert plano["classes_excluidas"] == [2, 3]
    assert plano["resolucao"] == 256

    classmap = json.loads(
        (DADOS / "classmap_1.json").read_text()
    )

    for ep in episodios:
        pid = int(ep["paciente"])
        z = int(ep["fatia"])
        assert pid in range(7, 30)
        assert z not in classmap["KID_R"][str(pid)]
        assert z not in classmap["KID_l"][str(pid)]

    random.seed(SEMENTE)
    np.random.seed(SEMENTE)
    torch.manual_seed(SEMENTE)
    torch.cuda.manual_seed_all(SEMENTE)

    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False

    pesos = ROOT / "pretrained_model/vmamba_tiny_v2.pth"
    assert pesos.is_file()

    config = {
        "dataset": "SABS",
        "modalidade": "CT",
        "fold": 0,
        "encoder": "VMamba-Tiny",
        "inicializacao": "pretrained",
        "pesos": str(pesos),
        "pesos_sha256": sha256(pesos),
        "plano_sha256": sha256(PLANO),
        "semente": SEMENTE,
        "passos": PASSOS,
        "resolucao": 256,
        "classes_excluidas": [2, 3],
        "loss": "CE + alinhamento",
        "otimizador": "AdamW",
        "lr": LR,
        "weight_decay": WD,
        "checkpoint_passos": [0, 50, 100, 200],
    }

    salvar_json(RUN / "config_treino.json", config)

    print("Construindo VMamba-Tiny...", flush=True)

    modelo = construir_modelo("tiny", repo=ROOT)
    estado_align = instalar(modelo)
    modelo.to("cuda")
    modelo.train()

    otimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=LR,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=WD
    )

    salvar_checkpoint(0, modelo, otimizador)

    inicio_total = time.time()
    log_path = RUN / "passos.jsonl"

    with log_path.open("x", buffering=1) as log:
        for passo, registro in enumerate(episodios, 1):
            inicio = time.time()

            support, fg, query, alvo = episodio(registro)

            otimizador.zero_grad(set_to_none=True)
            estado_align["ultimo"] = None

            retorno = modelo(
                [[support]],
                [[fg]],
                [[1 - fg]],
                [query],
                isval=False,
                val_wsize=2,
                show_viz=False
            )

            logits, alinhamento = retorno[:2]

            assert logits.shape == (1, 2, 256, 256)

            ce = F.cross_entropy(logits, alvo)
            loss = ce + alinhamento

            assert torch.isfinite(loss).item()

            loss.backward()

            grads = [
                p.grad for p in modelo.encoder.parameters()
                if p.grad is not None
            ]

            assert grads
            assert any(torch.count_nonzero(g).item() > 0 for g in grads)
            assert all(torch.isfinite(g).all().item() for g in grads)

            otimizador.step()

            torch.cuda.synchronize()

            dados_passo = {
                "passo": passo,
                "paciente": registro["paciente"],
                "fatia": registro["fatia"],
                "regiao": registro["regiao"],
                "ce": float(ce.detach()),
                "alinhamento": float(
                    alinhamento.detach()
                    if torch.is_tensor(alinhamento)
                    else alinhamento
                ),
                "loss": float(loss.detach()),
                "segundos": time.time() - inicio,
                "alinhamento_excecao": estado_align["ultimo"],
            }

            log.write(json.dumps(dados_passo) + "\n")

            if passo % 10 == 0 or passo == 1:
                print(
                    f"Passo {passo}/{PASSOS} | "
                    f"CE={dados_passo['ce']:.4f} | "
                    f"Align={dados_passo['alinhamento']:.4f} | "
                    f"Loss={dados_passo['loss']:.4f}",
                    flush=True
                )

            if passo in (50, 100, 200):
                salvar_checkpoint(passo, modelo, otimizador)

            if passo % 10 == 0 or passo == 1:
                salvar_json(RUN / "status.json", {
                    "estado": "executando",
                    "passo": passo,
                    "total": PASSOS,
                    "loss": dados_passo["loss"],
                    "tempo_decorrido_s": time.time() - inicio_total
                })

    salvar_json(RUN / "status.json", {
        "estado": "concluido",
        "passo": PASSOS,
        "total": PASSOS,
        "tempo_total_s": time.time() - inicio_total
    })

    print("TREINAMENTO CONCLUÍDO", flush=True)


if __name__ == "__main__":
    try:
        treinar()
    except Exception:
        erro = traceback.format_exc()
        print(erro, flush=True)

        salvar_json(RUN / "status.json", {
            "estado": "erro",
            "traceback": erro
        })
        raise
'''

# Verificar sintaxe antes de salvar
compile(codigo, str(SCRIPT_CT), "exec")

# Não substituir versões anteriores
if SCRIPT_CT.exists():
    if SCRIPT_CT.read_text() != codigo:
        raise RuntimeError(
            "O script já existe com conteúdo diferente. "
            "Não foi sobrescrito."
        )
    print("Script já existe e possui o mesmo conteúdo.")
else:
    SCRIPT_CT.write_text(codigo, encoding="utf-8")
    print("Script criado com sucesso.")

print("\n=== VERIFICAÇÃO ===")
print("Arquivo:", SCRIPT_CT)
print("Sintaxe Python: OK")
print("Passos previstos: 200")
print("Checkpoints: 0, 50, 100, 200")
print("Execução iniciada: NÃO")

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import json
import hashlib
from datetime import datetime, timezone

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = REPO / "exp/eft_ct_piloto200_v1/treinar_ct.py"
RUN = REPO / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"

LOG = RUN / "console.log"
LANCAMENTO = RUN / "lancamento.json"

assert SCRIPT.is_file(), "Script não encontrado"
assert (RUN / "plano_200.json").is_file(), "Plano não encontrado"

# Evitar iniciar outro treinamento sobre a mesma pasta
assert not LANCAMENTO.exists(), "Treinamento já foi lançado"
assert not (RUN / "checkpoint_0000.pt").exists(), "Checkpoint existente"
assert not (RUN / "passos.jsonl").exists(), "Logs anteriores existentes"
assert not (RUN / "status.json").exists(), "Status anterior existente"

# Registrar o código efetivamente utilizado
sha_script = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()

# Reservar a execução contra lançamentos duplicados
fd = os.open(
    str(LANCAMENTO),
    os.O_CREAT | os.O_EXCL | os.O_WRONLY,
    0o600
)

try:
    ambiente = os.environ.copy()
    ambiente["PYTHONUNBUFFERED"] = "1"

    with LOG.open("x", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(REPO),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=ambiente
        )

    registro = {
        "pid": processo.pid,
        "python": sys.executable,
        "script": str(SCRIPT),
        "script_sha256": sha_script,
        "inicio_utc": datetime.now(timezone.utc).isoformat(),
        "log": str(LOG),
        "modo": "detached",
        "passos_previstos": 200
    }

    with os.fdopen(fd, "w", encoding="utf-8") as f:
        json.dump(registro, f, indent=2)

except Exception:
    try:
        os.close(fd)
    except OSError:
        pass
    LANCAMENTO.unlink(missing_ok=True)
    raise

print("=== EFT-CT INICIADO ===")
print("PID:", processo.pid)
print("Python:", sys.executable)
print("Script:", SCRIPT.name)
print("Log:", LOG)
print("Passos previstos: 200")
print("Estado: lançamento solicitado; execução ainda não verificada")

In [ ]:
from pathlib import Path
import json
import subprocess

RUN = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/"
    "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
)

# 1. Identificar o processo
lancamento = json.loads(
    (RUN / "lancamento.json").read_text()
)

pid = lancamento["pid"]

proc = Path(f"/proc/{pid}")
ativo = False

if proc.exists():
    try:
        cmdline = (proc / "cmdline").read_bytes().replace(b"\x00", b" ")
        estado_proc = (proc / "stat").read_text().split(") ")[1][0]
        ativo = (
            b"treinar_ct.py" in cmdline
            and estado_proc not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

# 2. Ler estado informado pelo treinador
status_path = RUN / "status.json"

status = (
    json.loads(status_path.read_text())
    if status_path.exists()
    else {}
)

# 3. Ler progresso real
log_passos = RUN / "passos.jsonl"
passos = []

if log_passos.exists():
    for linha in log_passos.read_text().splitlines():
        try:
            passos.append(json.loads(linha))
        except json.JSONDecodeError:
            pass

ultimo_passo = passos[-1]["passo"] if passos else 0

# 4. Mostrar resumo
print("=" * 55)
print("EFT-CT | VMamba-Tiny | Felzenszwalb")
print("=" * 55)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado registrado:", status.get("estado", "ainda não registrado"))

total = 200
percentual = ultimo_passo / total * 100

print(f"\nPROGRESSO: {ultimo_passo}/{total} ({percentual:.1f}%)")

barra = (
    "█" * int(percentual // 5)
    + "░" * (20 - int(percentual // 5))
)
print(f"[{barra}]")

# 5. Últimas perdas
if passos:
    print("\n=== ÚLTIMOS 5 PASSOS ===")
    print("Passo | CE       | Alinh.   | Loss     | Segundos")

    for p in passos[-5:]:
        print(
            f"{p['passo']:5d} | "
            f"{p['ce']:.6f} | "
            f"{p['alinhamento']:.6f} | "
            f"{p['loss']:.6f} | "
            f"{p['segundos']:.2f}"
        )

    recentes = passos[-10:]
    media_loss = sum(p["loss"] for p in recentes) / len(recentes)

    print(f"\nLoss média (últimos {len(recentes)} passos): {media_loss:.6f}")

# 6. Checkpoints
checkpoints = sorted(RUN.glob("checkpoint_*.pt"))

print("\n=== CHECKPOINTS ===")
if checkpoints:
    for ckpt in checkpoints:
        tamanho = ckpt.stat().st_size / 1024**2
        print(f"{ckpt.name} ({tamanho:.1f} MB)")
else:
    print("Nenhum checkpoint encontrado")

# 7. Memória GPU usada pelo processo
print("\n=== GPU ===")

try:
    resultado = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,used_gpu_memory",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    linhas_gpu = [
        linha.strip()
        for linha in resultado.stdout.splitlines()
        if linha.strip().split(",")[0].strip() == str(pid)
    ]

    if linhas_gpu:
        print("Processo encontrado na GPU:", linhas_gpu[0])
    else:
        print("PID não encontrado entre os processos CUDA")

except Exception as e:
    print("Não foi possível consultar GPU:", str(e))

# 8. Diagnóstico final
print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NO TREINAMENTO")
    print(status.get("traceback", "")[-3000:])
elif status.get("estado") == "concluido":
    print("TREINAMENTO CONCLUÍDO")
elif ativo:
    print("Processo ativo; treinamento em andamento ou inicializando")
else:
    print("Processo não está ativo; conclusão ainda não confirmada")

# 9. Últimas mensagens do console
console = RUN / "console.log"

if console.exists():
    print("\n=== ÚLTIMAS MENSAGENS DO CONSOLE ===")
    for linha in console.read_text(
        errors="replace"
    ).splitlines()[-12:]:
        print(linha)

In [ ]:
import json
from pathlib import Path

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DADOS = REPO / "data/SABS/sabs_CT_normalized"

classmap = json.loads(
    (DADOS / "classmap_1.json").read_text()
)

SUPORTE = 6
QUERIES = list(range(6))

ORGAOS = {
    "Rim direito": "KID_R",
    "Rim esquerdo": "KID_l"
}

print("=== PROTOCOLO DE AVALIAÇÃO CT ===")
print("Paciente suporte:", SUPORTE)
print("Pacientes query:", QUERIES)

for nome, classe in ORGAOS.items():

    fatias_suporte = classmap[classe][str(SUPORTE)]

    print(f"\n=== {nome.upper()} ===")
    print("Fatias com órgão no suporte:", len(fatias_suporte))

    if fatias_suporte:
        print("Primeira fatia:", fatias_suporte[0])
        print("Última fatia:", fatias_suporte[-1])

    total = 0

    for pid in QUERIES:
        n = len(classmap[classe][str(pid)])
        total += n
        print(f"Query {pid}: {n} fatias com órgão")

    print("Total de fatias positivas:", total)

# Verificar checkpoints
RUN = REPO / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"

print("\n=== CHECKPOINTS PARA COMPARAÇÃO ===")

for passo in [0, 50, 100, 200]:
    arquivo = RUN / f"checkpoint_{passo:04d}.pt"
    print(f"Passo {passo}: {'OK' if arquivo.is_file() else 'AUSENTE'}")

In [ ]:
from pathlib import Path

ARQUIVOS = [
    REPO / "exp/eft_tiny_512_v1/avaliar_eft_direto_penultima512.py",
    REPO / "exp/eft_tiny_512_v1/eft_modelo_672.py",
    REPO / "validation.py",
]

TERMOS = [
    "isval=True",
    "model.eval(",
    "modelo.eval(",
    "load_state_dict",
    "argmax(",
    "softmax(",
    "val_wsize",
    "torch.no_grad",
    "inference_mode",
]

for arquivo in ARQUIVOS:
    print(f"\n{'=' * 65}")
    print(arquivo.relative_to(REPO))
    print(f"{'=' * 65}")

    if not arquivo.exists():
        print("Arquivo não encontrado")
        continue

    linhas = arquivo.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    indices = set()

    for i, linha in enumerate(linhas):
        if any(t.lower() in linha.lower() for t in TERMOS):
            indices.update(
                range(max(0, i - 5), min(len(linhas), i + 8))
            )

    for i in sorted(indices)[:150]:
        print(f"{i+1:04}: {linhas[i]}")

    if len(indices) > 150:
        print(f"... {len(indices) - 150} linhas omitidas")

In [ ]:
import os
import sys
import gc
import numpy as np
import torch
import SimpleITK as sitk

from pathlib import Path

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DADOS = REPO / "data/SABS/sabs_CT_normalized"
RUN = REPO / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"

MODULOS = REPO / "exp/eft_tiny_512_v1"

os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(MODULOS))

from eft_modelo_672 import construir_modelo

# Configuração
CLASSE = 2  # Rim direito
PID_SUPORTE = 6
PID_QUERY = 0
PASSO = 0

# Carregar volumes e máscaras
def carregar(pid):
    imagem = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
    )

    mascara = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"label_{pid}.nii.gz"))
    )

    assert imagem.shape == mascara.shape
    return imagem, mascara


img_s, mask_s = carregar(PID_SUPORTE)
img_q, mask_q = carregar(PID_QUERY)

# Selecionar fatia central com órgão presente
def selecionar_fatia(mascara, classe):
    positivas = np.where(
        np.any(mascara == classe, axis=(1, 2))
    )[0]

    assert len(positivas) > 0
    return int(positivas[len(positivas) // 2])


z_s = selecionar_fatia(mask_s, CLASSE)
z_q = selecionar_fatia(mask_q, CLASSE)

# Preparação igual à utilizada no EFT
def preparar_imagem(fatia):
    rgb = np.repeat(
        fatia.astype(np.float32)[..., None],
        3,
        axis=2
    )

    tensor = torch.from_numpy(
        np.ascontiguousarray(rgb)
    ).permute(2, 0, 1)[None].to(
        "cuda", dtype=torch.float32
    )

    media = tensor.new_tensor(
        [123.675, 116.28, 103.53]
    )[None, :, None, None]

    desvio = tensor.new_tensor(
        [58.395, 57.12, 57.375]
    )[None, :, None, None]

    return (tensor - media) / desvio


support = preparar_imagem(img_s[z_s])
query = preparar_imagem(img_q[z_q])

fg = torch.from_numpy(
    (mask_s[z_s] == CLASSE).astype(np.float32)
)[None].to("cuda")

gt = mask_q[z_q] == CLASSE

print("=== EPISÓDIO ===")
print("Órgão: Rim direito")
print("Suporte:", PID_SUPORTE, "fatia:", z_s)
print("Query:", PID_QUERY, "fatia:", z_q)
print("Pixels GT:", int(gt.sum()))

# Construir VMamba
modelo = construir_modelo("tiny", repo=REPO)

# Carregar checkpoint
checkpoint_path = RUN / f"checkpoint_{PASSO:04d}.pt"

# Checkpoint local criado pelo nosso treinador
ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert ckpt["passo"] == PASSO

modelo.load_state_dict(
    ckpt["modelo"],
    strict=True
)

modelo.to("cuda")
modelo.eval().requires_grad_(False)

print("\n=== INFERÊNCIA ALP ===")

with torch.no_grad():
    retorno = modelo(
        [[support]],
        [[fg]],
        [[1 - fg]],
        [query],
        isval=True,
        val_wsize=2,
        show_viz=False
    )

    logits = retorno[0]

    assert tuple(logits.shape) == (1, 2, 256, 256)
    assert torch.isfinite(logits).all().item()

    pred = (
        logits.argmax(dim=1)[0]
        .cpu()
        .numpy()
        == 1
    )

# Métricas
intersecao = np.logical_and(pred, gt).sum()
total_pred = pred.sum()
total_gt = gt.sum()

dice = (
    2 * intersecao / (total_pred + total_gt)
    if total_pred + total_gt > 0
    else 1.0
)

uniao = np.logical_or(pred, gt).sum()
iou = intersecao / uniao if uniao > 0 else 1.0

print("Logits:", tuple(logits.shape))
print("Pixels previstos:", int(total_pred))

print("\n=== RESULTADO ===")
print(f"Dice ALP: {dice:.6f}")
print(f"IoU ALP:  {iou:.6f}")

# Liberar GPU
del modelo, ckpt, retorno, logits
del support, query, fg
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display
import numpy as np

# Reutilizar a imagem, GT e previsão da célula 43
ct = np.clip(img_q[z_q], 0, 255).astype(np.uint8)

base = np.stack([ct] * 3, axis=-1)

# 1. Anotação real em verde
real = base.copy()
real[gt] = (
    0.45 * base[gt] +
    0.55 * np.array([0, 255, 0])
).astype(np.uint8)

# 2. Previsão em amarelo
previsto = base.copy()
previsto[pred] = (
    0.45 * base[pred] +
    0.55 * np.array([255, 220, 0])
).astype(np.uint8)

# 3. Mapa de erros
tp = pred & gt
fp = pred & ~gt
fn = ~pred & gt

erros = base.copy()
erros[tp] = [0, 255, 0]       # Verde: acerto
erros[fp] = [255, 60, 60]     # Vermelho: falso positivo
erros[fn] = [60, 140, 255]    # Azul: falso negativo

# Criar painel
imagens = [
    ("CT ORIGINAL", base),
    ("ANOTACAO REAL", real),
    ("PREVISAO ALP", previsto),
    ("MAPA DE ERROS", erros)
]

painel = Image.new("RGB", (1024, 290), "white")
draw = ImageDraw.Draw(painel)

for i, (titulo, array) in enumerate(imagens):
    painel.paste(Image.fromarray(array), (i * 256, 34))
    draw.text((i * 256 + 10, 10), titulo, fill="black")

display(painel)

# Métricas complementares
precisao = tp.sum() / max(pred.sum(), 1)
recall = tp.sum() / max(gt.sum(), 1)

print("=== DIAGNÓSTICO ===")
print(f"Precisão: {precisao:.4f}")
print(f"Recall:   {recall:.4f}")
print(f"TP: {tp.sum()}")
print(f"FP: {fp.sum()}")
print(f"FN: {fn.sum()}")

print("\nVerde = correto")
print("Vermelho = falso positivo")
print("Azul = falso negativo")

In [ ]:
import gc
import numpy as np
import torch
from PIL import Image, ImageDraw
from IPython.display import display

# Reutiliza os dados e as funções da célula 43.
# Nenhum treinamento será executado.

support = preparar_imagem(img_s[z_s])
query = preparar_imagem(img_q[z_q])

fg = torch.from_numpy(
    (mask_s[z_s] == CLASSE).astype(np.float32)
)[None].to("cuda")

gt = mask_q[z_q] == CLASSE

modelo = construir_modelo("tiny", repo=REPO)
modelo.to("cuda")
modelo.eval().requires_grad_(False)

resultados = {}

for passo in [0, 200]:
    print(f"\n=== CHECKPOINT {passo} ===")

    caminho = RUN / f"checkpoint_{passo:04d}.pt"

    ckpt = torch.load(
        caminho,
        map_location="cpu",
        weights_only=False
    )

    assert ckpt["passo"] == passo

    modelo.load_state_dict(
        ckpt["modelo"],
        strict=True
    )

    modelo.eval()

    with torch.inference_mode():
        retorno = modelo(
            [[support]],
            [[fg]],
            [[1 - fg]],
            [query],
            isval=True,
            val_wsize=2,
            show_viz=False
        )

        logits = retorno[0]

        assert logits.shape == (1, 2, 256, 256)

        predicao = (
            logits.argmax(1)[0]
            .cpu()
            .numpy() == 1
        )

    tp = int((predicao & gt).sum())
    fp = int((predicao & ~gt).sum())
    fn = int((~predicao & gt).sum())

    dice = 2 * tp / max(2 * tp + fp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    precisao = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    resultados[passo] = {
        "predicao": predicao.copy(),
        "dice": dice,
        "iou": iou,
        "precisao": precisao,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "fn": fn
    }

    print(f"Dice:      {dice:.4f}")
    print(f"IoU:       {iou:.4f}")
    print(f"Precisão:  {precisao:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"TP/FP/FN:  {tp}/{fp}/{fn}")

# Comparação
antes = resultados[0]
depois = resultados[200]

delta = (depois["dice"] - antes["dice"]) * 100

print("\n=== COMPARAÇÃO ===")
print(f"Dice inicial: {antes['dice']:.4f}")
print(f"Dice EFT 200: {depois['dice']:.4f}")
print(f"Variação: {delta:+.2f} pontos percentuais")

# Visualizar as mesmas imagens para ambos os checkpoints
ct = np.clip(img_q[z_q], 0, 255).astype(np.uint8)
base = np.stack([ct] * 3, axis=-1)

def sobrepor(mascara, cor):
    saida = base.copy()
    saida[mascara] = (
        0.4 * saida[mascara]
        + 0.6 * np.array(cor)
    ).astype(np.uint8)
    return saida

def mapa_erros(predicao):
    saida = base.copy()
    saida[predicao & gt] = [0, 255, 0]
    saida[predicao & ~gt] = [255, 60, 60]
    saida[~predicao & gt] = [60, 140, 255]
    return saida

imagens = [
    ("ANOTACAO REAL", sobrepor(gt, [0, 255, 0])),
    ("ANTES DO EFT", mapa_erros(antes["predicao"])),
    ("APOS EFT 200", mapa_erros(depois["predicao"]))
]

painel = Image.new("RGB", (768, 290), "white")
draw = ImageDraw.Draw(painel)

for i, (titulo, imagem) in enumerate(imagens):
    painel.paste(Image.fromarray(imagem), (256 * i, 34))
    draw.text((256 * i + 10, 10), titulo, fill="black")

display(painel)

print("Verde = acerto | Vermelho = falso positivo | Azul = falso negativo")

del modelo, ckpt, retorno, logits, support, query, fg
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from pathlib import Path

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
PASTA = REPO / "exp/eft_ct_piloto200_v1"
SCRIPT = PASTA / "avaliar_alp_ct.py"

codigo = r'''
import os
import sys
import csv
import json
import time
import traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import SimpleITK as sitk

ROOT = Path(__file__).resolve().parents[2]
MODULOS = ROOT / "exp/eft_tiny_512_v1"
DADOS = ROOT / "data/SABS/sabs_CT_normalized"
TREINO = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
SAIDA = TREINO / "avaliacao_alp_rins_positivas_f0"

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(MODULOS))

from eft_modelo_672 import construir_modelo

CHECKPOINTS = [0, 50, 100, 200]
ORGAOS = [(2, "KID_R"), (3, "KID_l")]
QUERIES = list(range(6))
SUPORTE = 6

def ler_volume(pid):
    imagem = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
    )
    label = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"label_{pid}.nii.gz"))
    )
    assert imagem.shape == label.shape
    assert imagem.shape[1:] == (256, 256)
    return imagem, label

def preparar_imagem(fatia):
    rgb = np.repeat(
        fatia.astype(np.float32)[..., None], 3, axis=2
    )
    tensor = torch.from_numpy(
        np.ascontiguousarray(rgb)
    ).permute(2, 0, 1)[None].to("cuda")

    media = tensor.new_tensor(
        [123.675, 116.28, 103.53]
    )[None, :, None, None]
    desvio = tensor.new_tensor(
        [58.395, 57.12, 57.375]
    )[None, :, None, None]

    return (tensor - media) / desvio

def escrever_status(**kwargs):
    temp = SAIDA / "status.json.tmp"
    temp.write_text(json.dumps(kwargs, indent=2))
    os.replace(temp, SAIDA / "status.json")

def main():
    assert torch.cuda.is_available()
    SAIDA.mkdir(parents=True, exist_ok=True)

    csv_fatias = SAIDA / "metricas_fatias.csv"
    assert not csv_fatias.exists(), (
        "Avaliação existente. Não sobrescrever."
    )

    classmap = json.loads(
        (DADOS / "classmap_1.json").read_text()
    )

    # Verificar todos os checkpoints antes de iniciar.
    for passo in CHECKPOINTS:
        assert (TREINO / f"checkpoint_{passo:04d}.pt").is_file()

    img_s, label_s = ler_volume(SUPORTE)

    suportes = {}

    for classe, nome in ORGAOS:
        indices = np.flatnonzero(
            np.any(label_s == classe, axis=(1, 2))
        )
        assert len(indices) > 0

        # Mesma política usada nas células 43 e 45.
        z = int(indices[len(indices) // 2])
        support = preparar_imagem(img_s[z])
        fg = torch.from_numpy(
            (label_s[z] == classe).astype(np.float32)
        )[None].to("cuda")

        suportes[classe] = (support, fg, z)
        print(f"Suporte {nome}: paciente 6, fatia {z}", flush=True)

    total_esperado = sum(
        len(classmap[nome][str(pid)])
        for _, nome in ORGAOS
        for pid in QUERIES
    ) * len(CHECKPOINTS)

    assert total_esperado == 1700, total_esperado

    modelo = construir_modelo("tiny", repo=ROOT)
    modelo.to("cuda").eval().requires_grad_(False)

    campos = [
        "checkpoint", "orgao", "paciente", "fatia",
        "dice", "iou", "precisao", "recall",
        "tp", "fp", "fn", "pixels_gt", "pixels_pred"
    ]

    concluidos = 0
    inicio = time.perf_counter()

    with csv_fatias.open(
        "x", newline="", encoding="utf-8", buffering=1
    ) as f:
        escritor = csv.DictWriter(f, fieldnames=campos)
        escritor.writeheader()

        for passo in CHECKPOINTS:
            checkpoint = torch.load(
                TREINO / f"checkpoint_{passo:04d}.pt",
                map_location="cpu",
                weights_only=False
            )
            assert checkpoint["passo"] == passo

            modelo.load_state_dict(
                checkpoint["modelo"], strict=True
            )
            del checkpoint
            modelo.eval()

            print(f"\nCHECKPOINT {passo}", flush=True)

            for classe, nome in ORGAOS:
                support, fg, _ = suportes[classe]

                for pid in QUERIES:
                    imagem, label = ler_volume(pid)
                    fatias = classmap[nome][str(pid)]

                    for z in fatias:
                        z = int(z)
                        gt = label[z] == classe
                        assert gt.any()

                        query = preparar_imagem(imagem[z])

                        with torch.inference_mode():
                            retorno = modelo(
                                [[support]], [[fg]], [[1 - fg]],
                                [query],
                                isval=True,
                                val_wsize=2,
                                show_viz=False
                            )
                            logits = retorno[0]
                            assert logits.shape == (1, 2, 256, 256)
                            assert torch.isfinite(logits).all().item()

                            pred = (
                                logits.argmax(dim=1)[0]
                                .cpu().numpy() == 1
                            )

                        tp = int((pred & gt).sum())
                        fp = int((pred & ~gt).sum())
                        fn = int((~pred & gt).sum())

                        registro = {
                            "checkpoint": passo,
                            "orgao": nome,
                            "paciente": pid,
                            "fatia": z,
                            "dice": 2 * tp / max(2*tp+fp+fn, 1),
                            "iou": tp / max(tp+fp+fn, 1),
                            "precisao": tp / max(tp+fp, 1),
                            "recall": tp / max(tp+fn, 1),
                            "tp": tp,
                            "fp": fp,
                            "fn": fn,
                            "pixels_gt": int(gt.sum()),
                            "pixels_pred": int(pred.sum())
                        }

                        escritor.writerow(registro)
                        concluidos += 1

                        if concluidos % 25 == 0:
                            escrever_status(
                                estado="executando",
                                concluidos=concluidos,
                                total=total_esperado,
                                checkpoint=passo,
                                orgao=nome,
                                paciente=pid
                            )
                            print(
                                f"{concluidos}/{total_esperado}",
                                flush=True
                            )

                        del query, retorno, logits

    # Resumir por paciente, depois entre os seis pacientes.
    agrupado = defaultdict(list)

    with csv_fatias.open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            chave = (
                int(r["checkpoint"]),
                r["orgao"],
                int(r["paciente"])
            )
            agrupado[chave].append(float(r["dice"]))

    linhas_pacientes = []

    for (passo, orgao, pid), valores in sorted(agrupado.items()):
        linhas_pacientes.append({
            "checkpoint": passo,
            "orgao": orgao,
            "paciente": pid,
            "n_fatias": len(valores),
            "dice_medio": float(np.mean(valores)),
            "dice_desvio_fatias": float(np.std(valores, ddof=1))
                if len(valores) > 1 else 0.0
        })

    with (SAIDA / "resumo_pacientes.csv").open(
        "w", newline="", encoding="utf-8"
    ) as f:
        escritor = csv.DictWriter(
            f, fieldnames=list(linhas_pacientes[0])
        )
        escritor.writeheader()
        escritor.writerows(linhas_pacientes)

    print("\n=== DICE MÉDIO ± DP ENTRE PACIENTES ===", flush=True)

    for passo in CHECKPOINTS:
        for _, orgao in ORGAOS:
            valores = [
                r["dice_medio"]
                for r in linhas_pacientes
                if r["checkpoint"] == passo and r["orgao"] == orgao
            ]

            assert len(valores) == 6

            print(
                f"{orgao} | passo {passo:3d} | "
                f"{np.mean(valores):.4f} ± "
                f"{np.std(valores, ddof=1):.4f}",
                flush=True
            )

    escrever_status(
        estado="concluido",
        concluidos=concluidos,
        total=total_esperado,
        segundos=time.perf_counter() - inicio
    )
    print("\nAVALIAÇÃO CONCLUÍDA", flush=True)

if __name__ == "__main__":
    try:
        main()
    except Exception:
        erro = traceback.format_exc()
        print(erro, flush=True)
        SAIDA.mkdir(parents=True, exist_ok=True)
        escrever_status(estado="erro", traceback=erro)
        raise
'''

compile(codigo, str(SCRIPT), "exec")

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "Já existe um avaliador diferente. Não sobrescrito."
    )
    print("Avaliador existente: conteúdo idêntico.")
else:
    SCRIPT.write_text(codigo, encoding="utf-8")
    print("Avaliador criado.")

print("\n=== VERIFICAÇÃO ===")
print("Arquivo:", SCRIPT)
print("Sintaxe: OK")
print("Checkpoints: 0, 50, 100, 200")
print("Órgãos: rim direito e esquerdo")
print("Pacientes query: 0–5")
print("Inferências previstas: 1.700")
print("Avaliação iniciada: NÃO")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import json
import sys
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = REPO / "exp/eft_ct_piloto200_v1/avaliar_alp_ct.py"

SAIDA = (
    REPO
    / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
    / "avaliacao_alp_rins_positivas_f0"
)

SAIDA.mkdir(parents=True, exist_ok=True)

LOG = SAIDA / "console.log"
LANCAMENTO = SAIDA / "lancamento.json"

assert SCRIPT.is_file(), "Script de avaliação não encontrado"

# Evitar iniciar duas avaliações na mesma pasta
assert not LANCAMENTO.exists(), "Avaliação já foi lançada"
assert not LOG.exists(), "Log anterior encontrado"
assert not (SAIDA / "status.json").exists(), "Status anterior encontrado"
assert not (SAIDA / "metricas_fatias.csv").exists(), "Resultados anteriores encontrados"

# Conferir checkpoints
TREINO = SAIDA.parent

for passo in [0, 50, 100, 200]:
    ckpt = TREINO / f"checkpoint_{passo:04d}.pt"
    assert ckpt.is_file(), f"Checkpoint ausente: {ckpt}"

sha_script = hashlib.sha256(
    SCRIPT.read_bytes()
).hexdigest()

# Reservar execução para impedir lançamentos duplicados
fd = os.open(
    str(LANCAMENTO),
    os.O_CREAT | os.O_EXCL | os.O_WRONLY,
    0o600
)

try:
    ambiente = os.environ.copy()
    ambiente["PYTHONUNBUFFERED"] = "1"

    with LOG.open("x", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(REPO),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=ambiente
        )

    registro = {
        "pid": processo.pid,
        "python": sys.executable,
        "script": str(SCRIPT),
        "script_sha256": sha_script,
        "inicio_utc": datetime.now(timezone.utc).isoformat(),
        "log": str(LOG),
        "modo": "detached",
        "inferencias_previstas": 1700,
        "checkpoints": [0, 50, 100, 200],
        "orgaos": ["KID_R", "KID_l"]
    }

    with os.fdopen(fd, "w", encoding="utf-8") as f:
        json.dump(registro, f, indent=2)

except Exception:
    try:
        os.close(fd)
    except OSError:
        pass
    LANCAMENTO.unlink(missing_ok=True)
    raise

print("=== AVALIAÇÃO ALP-CT LANÇADA ===")
print("PID:", processo.pid)
print("Python:", sys.executable)
print("Script:", SCRIPT.name)
print("Log:", LOG)
print("Inferências previstas: 1700")
print("Checkpoints: 0, 50, 100, 200")
print("Estado: lançamento solicitado; execução ainda não verificada")

In [ ]:
from pathlib import Path
from collections import Counter
from statistics import mean, stdev
import subprocess
import json
import csv

RUN = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/"
    "reproducao/eft_ct_felzenszwalb_fold0_piloto200/"
    "avaliacao_alp_rins_positivas_f0"
)

TOTAL = 1700

def ler_json(caminho):
    if not caminho.exists():
        return {}
    try:
        return json.loads(caminho.read_text())
    except (OSError, json.JSONDecodeError):
        return {}

def processo_ativo(pid):
    if not pid:
        return False

    proc = Path(f"/proc/{pid}")
    if not proc.exists():
        return False

    try:
        comando = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        return (
            b"avaliar_alp_ct.py" in comando
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        return False

lancamento = ler_json(RUN / "lancamento.json")
status = ler_json(RUN / "status.json")

pid = lancamento.get("pid")
ativo = processo_ativo(pid)

# Ler os resultados efetivamente gravados no CSV
registros = []
arquivo_metricas = RUN / "metricas_fatias.csv"

if arquivo_metricas.exists():
    with arquivo_metricas.open(
        newline="", encoding="utf-8"
    ) as f:
        registros = list(csv.DictReader(f))

concluidas = len(registros)
percentual = 100 * concluidas / TOTAL

print("=" * 62)
print("AVALIAÇÃO ALP-CT | VMamba-Tiny | 1.700 INFERÊNCIAS")
print("=" * 62)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))

print(f"\nPROGRESSO: {concluidas}/{TOTAL} ({percentual:.1f}%)")

blocos = min(30, int(percentual * 30 / 100))
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

# Progresso por checkpoint
contagem = Counter(
    int(r["checkpoint"]) for r in registros
)

print("\n=== PROGRESSO POR CHECKPOINT ===")
print("Checkpoint | Concluídas | Total")
print("-" * 37)

for passo in [0, 50, 100, 200]:
    n = contagem[passo]
    print(f"{passo:10d} | {n:10d} | 425")

# Últimas inferências
if registros:
    print("\n=== ÚLTIMAS 5 INFERÊNCIAS ===")
    print("Ckpt | Órgão  | Paciente | Fatia | Dice")
    print("-" * 48)

    for r in registros[-5:]:
        print(
            f"{int(r['checkpoint']):4d} | "
            f"{r['orgao']:6s} | "
            f"{int(r['paciente']):8d} | "
            f"{int(r['fatia']):5d} | "
            f"{float(r['dice']):.4f}"
        )

# Mostrar resultados finais quando disponíveis
resumo = RUN / "resumo_pacientes.csv"

if resumo.exists():
    with resumo.open(
        newline="", encoding="utf-8"
    ) as f:
        linhas = list(csv.DictReader(f))

    print("\n=== RESULTADO: DICE MÉDIO ± DESVIO PADRÃO ===")
    print("Órgão      | Ckpt 0       | Ckpt 50      | Ckpt 100     | Ckpt 200")
    print("-" * 83)

    for orgao in ["KID_R", "KID_l"]:
        valores_formatados = []

        for passo in [0, 50, 100, 200]:
            valores = [
                float(r["dice_medio"])
                for r in linhas
                if (
                    r["orgao"] == orgao
                    and int(r["checkpoint"]) == passo
                )
            ]

            if len(valores) == 6:
                valores_formatados.append(
                    f"{mean(valores):.3f}±{stdev(valores):.3f}"
                )
            else:
                valores_formatados.append("incompleto")

        print(
            f"{orgao:10s} | "
            + " | ".join(f"{v:13s}" for v in valores_formatados)
        )

# Situação na GPU
print("\n=== GPU ===")
try:
    resultado = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,used_gpu_memory",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    encontrado = False

    for linha in resultado.stdout.splitlines():
        partes = linha.split(",")

        if partes and partes[0].strip() == str(pid):
            print("Processo na GPU:", linha.strip())
            encontrado = True

    if not encontrado:
        print("Processo não encontrado na GPU")

except Exception as e:
    print("Consulta indisponível:", str(e))

# Diagnóstico
print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "")[-3000:])

elif status.get("estado") == "concluido":
    print("AVALIAÇÃO CONCLUÍDA")

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo; verificar log e status")

# Console
console = RUN / "console.log"

if console.exists():
    print("\n=== ÚLTIMAS MENSAGENS ===")
    for linha in console.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()[-12:]:
        print(linha)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import json
import sys
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT_BASE = REPO / "exp/eft_ct_piloto200_v1/treinar_ct.py"
SCRIPT = REPO / "exp/eft_ct_piloto200_v1/continuar_10000.py"

ORIGEM = REPO / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
RUN = REPO / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"

AVALIACAO = ORIGEM / "avaliacao_alp_rins_positivas_f0"

assert SCRIPT_BASE.is_file()
assert (ORIGEM / "checkpoint_0200.pt").is_file()
assert (ORIGEM / "plano_200.json").is_file()

codigo = r'''
import os
import sys
import json
import time
import random
import hashlib
import shutil
import traceback
import importlib.util
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import SimpleITK as sitk

ROOT = Path(__file__).resolve().parents[2]

BASE = ROOT / "exp/eft_ct_piloto200_v1/treinar_ct.py"
ORIGEM = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
RUN = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"
DADOS = ROOT / "data/SABS/sabs_CT_normalized"
AVALIACAO = ORIGEM / "avaliacao_alp_rins_positivas_f0"

os.chdir(ROOT)

spec = importlib.util.spec_from_file_location("eft_ct_base", BASE)
base = importlib.util.module_from_spec(spec)
spec.loader.exec_module(base)

PASSO_INICIAL = 200
TOTAL = 10000
SEMENTE = 1234

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def json_atomico(path, valor):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(
        json.dumps(valor, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(temp, path)

def processo_avaliacao_ativo():
    arquivo = AVALIACAO / "lancamento.json"
    if not arquivo.exists():
        return False

    pid = json.loads(arquivo.read_text())["pid"]
    proc = Path(f"/proc/{pid}")

    if not proc.exists():
        return False

    try:
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]
        return b"avaliar_alp_ct.py" in cmd and estado not in ("Z", "X")
    except (OSError, IndexError):
        return False

def aguardar_gpu():
    if processo_avaliacao_ativo():
        print("Aguardando a avaliação ALP liberar a GPU...", flush=True)

    while processo_avaliacao_ativo():
        json_atomico(RUN / "status.json", {
            "estado": "aguardando_avaliacao",
            "passo": 200,
            "total": TOTAL
        })
        time.sleep(10)

    print("GPU liberada pela avaliação anterior.", flush=True)

def construir_plano():
    plano200 = json.loads(
        (ORIGEM / "plano_200.json").read_text()
    )["episodios"]

    assert len(plano200) == 200

    mapa = json.loads(
        (DADOS / "classmap_1.json").read_text()
    )

    utilizados = {
        (int(r["paciente"]), int(r["fatia"]), int(r["regiao"]))
        for r in plano200
    }

    candidatos = []

    for pid in range(7, 30):
        caminho = DADOS / f"superpix-MIDDLE_{pid}.nii.gz"

        volume = sitk.GetArrayFromImage(
            sitk.ReadImage(str(caminho))
        )

        excluidas = (
            set(mapa["KID_R"][str(pid)])
            | set(mapa["KID_l"][str(pid)])
        )

        for z in range(volume.shape[0]):
            if z in excluidas:
                continue

            ids, areas = np.unique(
                volume[z], return_counts=True
            )

            for regiao, area in zip(ids, areas):
                regiao = int(regiao)
                area = int(area)

                if regiao <= 0 or not (400 <= area <= 10000):
                    continue

                if (pid, z, regiao) in utilizados:
                    continue

                candidatos.append({
                    "paciente": pid,
                    "fatia": z,
                    "regiao": regiao,
                    "area": area
                })

    assert len(candidatos) >= 9800

    novos = random.Random(SEMENTE).sample(
        candidatos, 9800
    )

    plano = {
        "metodo": "EFT-CT Felzenszwalb",
        "fold": 0,
        "resolucao": 256,
        "semente": SEMENTE,
        "passos": TOTAL,
        "primeiros_200": str(ORIGEM / "plano_200.json"),
        "episodios": plano200 + novos
    }

    assert len(plano["episodios"]) == TOTAL

    json_atomico(RUN / "plano_10000.json", plano)

    print(
        f"Plano: 200 episódios anteriores + "
        f"{len(novos)} novos.",
        flush=True
    )

    return plano["episodios"]

def salvar(passo, modelo, otimizador):
    estado = {
        "passo": passo,
        "modelo": modelo.state_dict(),
        "otimizador": otimizador.state_dict(),
        "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(),
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all()
    }

    temp = RUN / "checkpoint_latest.pt.tmp"
    latest = RUN / "checkpoint_latest.pt"

    torch.save(estado, temp)
    os.replace(temp, latest)

    if passo % 500 == 0 or passo == TOTAL:
        destino = RUN / f"checkpoint_{passo:05d}.pt"
        temporario = destino.with_suffix(".pt.tmp")
        shutil.copy2(latest, temporario)
        os.replace(temporario, destino)
        print(f"Checkpoint preservado: {passo}", flush=True)

def executar():
    RUN.mkdir(parents=True, exist_ok=True)

    # Preservar a avaliação: não disputar a GPU com ela.
    aguardar_gpu()

    # Impedir reinício acidental sobre resultados existentes.
    assert not (RUN / "passos.jsonl").exists()
    assert not (RUN / "checkpoint_latest.pt").exists()

    episodios = construir_plano()

    origem_ckpt = ORIGEM / "checkpoint_0200.pt"

    config = {
        "dataset": "SABS CT",
        "fold": 0,
        "encoder": "VMamba-Tiny",
        "inicio": "continuação exata do checkpoint 200",
        "checkpoint_origem": str(origem_ckpt),
        "checkpoint_sha256": sha(origem_ckpt),
        "script_base_sha256": sha(BASE),
        "plano_sha256": sha(RUN / "plano_10000.json"),
        "passos_totais": TOTAL,
        "passos_adicionais": 9800,
        "resolucao": 256,
        "otimizador": "AdamW",
        "lr": 1e-5,
        "weight_decay": 0.01,
        "loss": "cross entropy + alinhamento",
        "classes_excluidas": [2, 3],
        "checkpoints_preservados": "500, 1000, ..., 10000"
    }

    json_atomico(RUN / "config.json", config)
    shutil.copy2(BASE, RUN / "treinador_base.py")

    print("Construindo VMamba-Tiny...", flush=True)

    modelo = base.construir_modelo("tiny", repo=ROOT)
    estado_align = base.instalar(modelo)
    modelo.to("cuda")

    otimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=1e-5,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0.01
    )

    checkpoint = torch.load(
        origem_ckpt,
        map_location="cpu",
        weights_only=False
    )

    assert checkpoint["passo"] == PASSO_INICIAL

    modelo.load_state_dict(
        checkpoint["modelo"],
        strict=True
    )

    otimizador.load_state_dict(
        checkpoint["otimizador"]
    )

    random.setstate(checkpoint["rng_python"])
    np.random.set_state(checkpoint["rng_numpy"])
    torch.set_rng_state(checkpoint["rng_torch"])
    torch.cuda.set_rng_state_all(checkpoint["rng_cuda"])

    del checkpoint

    modelo.train()

    print(
        "Checkpoint 200 restaurado: "
        "pesos, otimizador e RNG.",
        flush=True
    )

    inicio = time.perf_counter()

    with (RUN / "passos.jsonl").open(
        "x", encoding="utf-8", buffering=1
    ) as log:

        for passo in range(201, TOTAL + 1):
            registro = episodios[passo - 1]
            tempo_passo = time.perf_counter()

            support, fg, query, alvo = base.episodio(registro)

            otimizador.zero_grad(set_to_none=True)
            estado_align["ultimo"] = None

            retorno = modelo(
                [[support]],
                [[fg]],
                [[1 - fg]],
                [query],
                isval=False,
                val_wsize=2,
                show_viz=False
            )

            logits, alinhamento = retorno[:2]

            assert logits.shape == (1, 2, 256, 256)

            ce = F.cross_entropy(logits, alvo)
            loss = ce + alinhamento

            assert torch.isfinite(loss).item()

            loss.backward()

            # Gradientes conferidos em intervalos regulares.
            if passo == 201 or passo % 100 == 0:
                grads = [
                    p.grad
                    for p in modelo.encoder.parameters()
                    if p.grad is not None
                ]
                assert grads
                assert all(
                    torch.isfinite(g).all().item()
                    for g in grads
                )
                assert any(
                    torch.count_nonzero(g).item() > 0
                    for g in grads
                )

            otimizador.step()

            if passo % 25 == 0:
                torch.cuda.synchronize()

            linha = {
                "passo": passo,
                "paciente": registro["paciente"],
                "fatia": registro["fatia"],
                "regiao": registro["regiao"],
                "ce": float(ce.detach()),
                "alinhamento": float(
                    alinhamento.detach()
                    if torch.is_tensor(alinhamento)
                    else alinhamento
                ),
                "loss": float(loss.detach()),
                "segundos": time.perf_counter() - tempo_passo
            }

            log.write(json.dumps(linha) + "\n")

            if passo % 25 == 0 or passo == 201:
                json_atomico(RUN / "status.json", {
                    "estado": "executando",
                    "passo": passo,
                    "total": TOTAL,
                    "loss": linha["loss"],
                    "segundos_decorridos": (
                        time.perf_counter() - inicio
                    )
                })

            if passo % 100 == 0:
                salvar(passo, modelo, otimizador)
                print(
                    f"Passo {passo}/{TOTAL} | "
                    f"loss={linha['loss']:.4f}",
                    flush=True
                )

            del retorno, logits, alinhamento
            del ce, loss, support, fg, query, alvo

    json_atomico(RUN / "status.json", {
        "estado": "concluido",
        "passo": TOTAL,
        "total": TOTAL,
        "segundos": time.perf_counter() - inicio
    })

    print("TREINAMENTO 10000 CONCLUÍDO", flush=True)

if __name__ == "__main__":
    try:
        executar()
    except Exception:
        erro = traceback.format_exc()
        print(erro, flush=True)
        RUN.mkdir(parents=True, exist_ok=True)
        json_atomico(
            RUN / "status.json",
            {"estado": "erro", "traceback": erro}
        )
        raise
'''

compile(codigo, str(SCRIPT), "exec")

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "Script existente é diferente. Não sobrescrito."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

RUN.mkdir(parents=True, exist_ok=True)

LOG = RUN / "console.log"
LANCAMENTO = RUN / "lancamento.json"

assert not LANCAMENTO.exists(), "Execução já lançada."
assert not LOG.exists(), "Log anterior encontrado."
assert not (RUN / "passos.jsonl").exists()
assert not (RUN / "checkpoint_latest.pt").exists()

# Reserva exclusiva contra lançamentos duplicados
fd = os.open(
    str(LANCAMENTO),
    os.O_CREAT | os.O_EXCL | os.O_WRONLY,
    0o600
)

try:
    ambiente = os.environ.copy()
    ambiente["PYTHONUNBUFFERED"] = "1"

    with LOG.open("x", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(REPO),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=ambiente
        )

    registro = {
        "pid": processo.pid,
        "inicio_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "python": sys.executable,
        "script": str(SCRIPT),
        "script_sha256": hashlib.sha256(
            SCRIPT.read_bytes()
        ).hexdigest(),
        "checkpoint_inicial": 200,
        "objetivo": 10000,
        "modo": "detached"
    }

    with os.fdopen(fd, "w") as f:
        json.dump(registro, f, indent=2)

except Exception:
    try:
        os.close(fd)
    except OSError:
        pass
    LANCAMENTO.unlink(missing_ok=True)
    raise

print("=== EFT-CT 10.000 PASSOS ===")
print("PID:", processo.pid)
print("Script:", SCRIPT)
print("Resultados:", RUN)
print("Continuação: passo 200 → 10000")
print("Checkpoints: a cada 100 passos")
print("Preservação: a cada 500 passos")
print("Modo: detached")
print("Início automático após a avaliação liberar a GPU")
print("Lançamento solicitado; execução ainda não verificada")

In [ ]:
from pathlib import Path
from collections import deque
from statistics import mean
import json
import subprocess

RUN = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/"
    "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"
)

INICIO = 200
TOTAL = 10000

def ler_json(arquivo):
    if not arquivo.exists():
        return {}
    try:
        return json.loads(arquivo.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

def verificar_processo(pid):
    if not pid:
        return False

    proc = Path(f"/proc/{pid}")

    try:
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        return (
            b"continuar_10000.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        return False

lancamento = ler_json(RUN / "lancamento.json")
status = ler_json(RUN / "status.json")

pid = lancamento.get("pid")
ativo = verificar_processo(pid)

# Ler as últimas 20 métricas sem carregar tudo na memória
ultimos = deque(maxlen=20)
quantidade = 0

arquivo_passos = RUN / "passos.jsonl"

if arquivo_passos.exists():
    with arquivo_passos.open(encoding="utf-8") as f:
        for linha in f:
            try:
                registro = json.loads(linha)
                quantidade += 1
                ultimos.append(registro)
            except json.JSONDecodeError:
                pass

passo = ultimos[-1]["passo"] if ultimos else INICIO
restantes = max(0, TOTAL - passo)
percentual = passo / TOTAL * 100

print("=" * 62)
print("EFT-CT | VMamba-Tiny | 10.000 PASSOS")
print("=" * 62)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))

print(f"\nPROGRESSO TOTAL: {passo:,}/{TOTAL:,} ({percentual:.2f}%)")
print(f"Passos desta continuação: {quantidade:,}/9.800")
print(f"Passos restantes: {restantes:,}")

blocos = min(30, int(percentual * 30 / 100))
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

# Últimas perdas
if ultimos:
    print("\n=== ÚLTIMOS 5 PASSOS ===")
    print("Passo | CE       | Alinh.   | Loss     | Segundos")
    print("-" * 58)

    for p in list(ultimos)[-5:]:
        print(
            f"{p['passo']:5d} | "
            f"{p['ce']:.6f} | "
            f"{p['alinhamento']:.6f} | "
            f"{p['loss']:.6f} | "
            f"{p['segundos']:.2f}"
        )

    media_loss = mean(p["loss"] for p in ultimos)
    media_tempo = mean(p["segundos"] for p in ultimos)

    print(f"\nLoss média (últimos {len(ultimos)}): {media_loss:.6f}")
    print(f"Tempo médio por passo: {media_tempo:.2f} s")

    if ativo and restantes:
        minutos = restantes * media_tempo / 60
        print(f"Tempo restante estimado: {minutos:.1f} min")
        print("(Estimativa baseada nos últimos passos)")

# Checkpoints
print("\n=== CHECKPOINTS ===")

checkpoints = sorted(RUN.glob("checkpoint_*.pt"))

if checkpoints:
    print("Checkpoints preservados:", len(checkpoints))
    print("Mais recente:", checkpoints[-1].name)
else:
    print("Nenhum checkpoint histórico salvo ainda")

latest = RUN / "checkpoint_latest.pt"

if latest.exists():
    print(
        "Checkpoint de retomada:",
        f"{latest.stat().st_size / 1024**2:.1f} MB"
    )
else:
    print("Checkpoint de retomada: ainda não criado")

# GPU
print("\n=== GPU ===")

try:
    resultado = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,used_gpu_memory",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    encontrado = False

    for linha in resultado.stdout.splitlines():
        partes = linha.split(",")

        if partes[0].strip() == str(pid):
            print("Processo encontrado:", linha.strip())
            encontrado = True

    if not encontrado:
        print("PID não encontrado na GPU")

except Exception as e:
    print("Consulta indisponível:", e)

# Diagnóstico
print("\n=== DIAGNÓSTICO ===")

estado = status.get("estado")

if estado == "erro":
    print("ERRO NO TREINAMENTO")
    print(status.get("traceback", "")[-3500:])

elif estado == "concluido":
    print("TREINAMENTO CONCLUÍDO")

elif estado == "aguardando_avaliacao":
    print("Aguardando término da avaliação anterior")

elif ativo:
    print("Processo ativo: treinamento ou inicialização em andamento")

else:
    print("Processo inativo sem conclusão confirmada")
    print("Verifique as mensagens abaixo")

# Últimas mensagens de console
console = RUN / "console.log"

if console.exists():
    print("\n=== ÚLTIMAS MENSAGENS ===")

    linhas = deque(maxlen=15)

    with console.open(
        encoding="utf-8",
        errors="replace"
    ) as f:
        for linha in f:
            linhas.append(linha.rstrip())

    for linha in linhas:
        print(linha)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import subprocess
import json
import sys
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

PASTA_SCRIPTS = REPO / "exp/eft_ct_piloto200_v1"

ORIGINAL = PASTA_SCRIPTS / "avaliar_alp_ct.py"
NOVO = PASTA_SCRIPTS / "avaliar_alp_ct_10000.py"

PILOTO = REPO / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
TREINO = REPO / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"

SAIDA = TREINO / "avaliacao_alp_rins_positivas_f0_7ckpts"

PASSOS = [0, 200, 500, 1000, 2000, 5000, 10000]

assert ORIGINAL.is_file()

# Confirmar checkpoints antes de criar a avaliação
for passo in PASSOS:
    if passo in (0, 200):
        arquivo = PILOTO / f"checkpoint_{passo:04d}.pt"
    else:
        arquivo = TREINO / f"checkpoint_{passo:05d}.pt"

    assert arquivo.is_file(), f"Checkpoint ausente: {arquivo}"

print("Todos os 7 checkpoints encontrados.")

codigo = ORIGINAL.read_text(encoding="utf-8")

alteracoes = [
    (
        'TREINO = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"',
        '''TREINO = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"
PILOTO = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"

def arquivo_checkpoint(passo):
    if passo in (0, 200):
        return PILOTO / f"checkpoint_{passo:04d}.pt"
    return TREINO / f"checkpoint_{passo:05d}.pt"'''
    ),
    (
        'SAIDA = TREINO / "avaliacao_alp_rins_positivas_f0"',
        'SAIDA = TREINO / "avaliacao_alp_rins_positivas_f0_7ckpts"'
    ),
    (
        'CHECKPOINTS = [0, 50, 100, 200]',
        'CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]'
    ),
    (
        'assert total_esperado == 1700, total_esperado',
        'assert total_esperado == 2975, total_esperado'
    ),
    (
        'assert (TREINO / f"checkpoint_{passo:04d}.pt").is_file()',
        'assert arquivo_checkpoint(passo).is_file()'
    ),
    (
        'TREINO / f"checkpoint_{passo:04d}.pt",',
        'arquivo_checkpoint(passo),'
    )
]

for antigo, novo in alteracoes:
    ocorrencias = codigo.count(antigo)
    assert ocorrencias == 1, (
        f"Esperada uma ocorrência, encontradas {ocorrencias}: {antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

# Validar sintaxe
compile(codigo, str(NOVO), "exec")

# Criar sem sobrescrever modificações anteriores
if NOVO.exists():
    assert NOVO.read_text(encoding="utf-8") == codigo, (
        "Existe um script diferente nesse caminho."
    )
else:
    NOVO.write_text(codigo, encoding="utf-8")

SAIDA.mkdir(parents=True, exist_ok=True)

LOG = SAIDA / "console.log"
LANCAMENTO = SAIDA / "lancamento.json"

for arquivo in [
    LANCAMENTO,
    LOG,
    SAIDA / "status.json",
    SAIDA / "metricas_fatias.csv",
]:
    assert not arquivo.exists(), (
        f"Avaliação anterior encontrada: {arquivo}"
    )

# Impedir lançamentos duplicados
fd = os.open(
    str(LANCAMENTO),
    os.O_CREAT | os.O_EXCL | os.O_WRONLY,
    0o600
)

try:
    ambiente = os.environ.copy()
    ambiente["PYTHONUNBUFFERED"] = "1"

    with LOG.open("x", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(NOVO)],
            cwd=str(REPO),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=ambiente
        )

    registro = {
        "pid": processo.pid,
        "script": str(NOVO),
        "script_sha256": hashlib.sha256(
            NOVO.read_bytes()
        ).hexdigest(),
        "python": sys.executable,
        "inicio_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "checkpoints": PASSOS,
        "inferencias_previstas": 2975,
        "modo": "detached"
    }

    with os.fdopen(fd, "w", encoding="utf-8") as f:
        json.dump(registro, f, indent=2)

except Exception:
    try:
        os.close(fd)
    except OSError:
        pass
    LANCAMENTO.unlink(missing_ok=True)
    raise

print("\n=== AVALIAÇÃO EFT-CT 10.000 ===")
print("PID:", processo.pid)
print("Script:", NOVO)
print("Resultados:", SAIDA)
print("Checkpoints:", PASSOS)
print("Inferências previstas: 2975")
print("Modo: detached")
print("Lançamento solicitado; execução ainda não verificada")

In [ ]:
from pathlib import Path
from collections import Counter, deque
from statistics import mean, stdev
import subprocess
import json
import csv

RUN = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/"
    "reproducao/eft_ct_felzenszwalb_fold0_10000_v1/"
    "avaliacao_alp_rins_positivas_f0_7ckpts"
)

CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]
TOTAL = 2975
POR_CHECKPOINT = 425

def ler_json(arquivo):
    try:
        return json.loads(arquivo.read_text(encoding="utf-8"))
    except (FileNotFoundError, OSError, json.JSONDecodeError):
        return {}

def esta_ativo(pid):
    if pid is None:
        return False

    proc = Path(f"/proc/{pid}")

    try:
        comando = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        return (
            b"avaliar_alp_ct_10000.py" in comando
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        return False

lancamento = ler_json(RUN / "lancamento.json")
status = ler_json(RUN / "status.json")

pid = lancamento.get("pid")
ativo = esta_ativo(pid)

# Ler métricas gravadas até o momento
arquivo = RUN / "metricas_fatias.csv"

contagem = Counter()
ultimos = deque(maxlen=5)
concluidas = 0

if arquivo.exists():
    with arquivo.open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            try:
                passo = int(r["checkpoint"])
                contagem[passo] += 1
                concluidas += 1
                ultimos.append(r)
            except (KeyError, ValueError):
                pass

percentual = 100 * concluidas / TOTAL

print("=" * 66)
print("AVALIAÇÃO EFT-CT | VMamba-Tiny | 7 CHECKPOINTS")
print("=" * 66)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado registrado:", status.get("estado", "inicializando"))

print(f"\nPROGRESSO: {concluidas:,}/{TOTAL:,} ({percentual:.2f}%)")

blocos = min(30, int(percentual * 30 / 100))
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

print("\n=== PROGRESSO POR CHECKPOINT ===")
print("Checkpoint | Concluídas | Total | Progresso")
print("-" * 53)

for passo in CHECKPOINTS:
    n = contagem[passo]
    pct = 100 * n / POR_CHECKPOINT

    print(
        f"{passo:10d} | "
        f"{n:10d} | "
        f"{POR_CHECKPOINT:5d} | "
        f"{pct:6.1f}%"
    )

if ultimos:
    print("\n=== ÚLTIMAS 5 INFERÊNCIAS ===")
    print("Ckpt  | Órgão | Paciente | Fatia | Dice")
    print("-" * 49)

    for r in ultimos:
        print(
            f"{int(r['checkpoint']):5d} | "
            f"{r['orgao']:5s} | "
            f"{int(r['paciente']):8d} | "
            f"{int(r['fatia']):5d} | "
            f"{float(r['dice']):.4f}"
        )

# Resultados finais, quando disponíveis
resumo = RUN / "resumo_pacientes.csv"

if resumo.exists():
    with resumo.open(newline="", encoding="utf-8") as f:
        linhas = list(csv.DictReader(f))

    print("\n=== DICE MÉDIO ± DP ENTRE PACIENTES ===")
    print(
        "Checkpoint | Rim direito    | Rim esquerdo    "
        "| Média dos órgãos"
    )
    print("-" * 72)

    for passo in CHECKPOINTS:
        medias = []
        formatados = []

        for orgao in ["KID_R", "KID_l"]:
            valores = [
                float(r["dice_medio"])
                for r in linhas
                if int(r["checkpoint"]) == passo
                and r["orgao"] == orgao
            ]

            if len(valores) == 6:
                m = mean(valores)
                dp = stdev(valores)
                medias.append(m)
                formatados.append(f"{m:.4f} ± {dp:.4f}")
            else:
                formatados.append("incompleto")

        media_orgaos = (
            f"{mean(medias):.4f}"
            if len(medias) == 2
            else "incompleto"
        )

        print(
            f"{passo:10d} | "
            f"{formatados[0]:14s} | "
            f"{formatados[1]:14s} | "
            f"{media_orgaos}"
        )

# GPU
print("\n=== GPU ===")

try:
    resultado = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,used_gpu_memory",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    encontrado = False

    for linha in resultado.stdout.splitlines():
        partes = linha.split(",")

        if partes[0].strip() == str(pid):
            print("Processo na GPU:", linha.strip(), "MiB")
            encontrado = True

    if not encontrado:
        print("PID não encontrado entre os processos CUDA")

except Exception as e:
    print("Consulta GPU indisponível:", e)

# Diagnóstico
print("\n=== DIAGNÓSTICO ===")

estado = status.get("estado")

if estado == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "")[-3500:])

elif estado == "concluido":
    print("AVALIAÇÃO CONCLUÍDA")
    print("Inferências registradas:", concluidas)

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada")

# Console
console = RUN / "console.log"

if console.exists():
    print("\n=== ÚLTIMAS MENSAGENS ===")

    linhas_console = deque(maxlen=15)

    with console.open(encoding="utf-8", errors="replace") as f:
        for linha in f:
            linhas_console.append(linha.rstrip())

    for linha in linhas_console:
        print(linha)

In [ ]:
from pathlib import Path

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ARQUIVO = (
    REPO
    / "exp/eft_tiny_512_v1"
    / "avaliar_eft_direto_penultima512.py"
)

assert ARQUIVO.exists()

linhas = ARQUIVO.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

termos = [
    "ALPNetWrapper",
    "ProtoSAM(",
    "cfg_sam",
    "sam_checkpoint",
    "sam_model_type",
    "sam_type",
    "query_image=",
    "coarse_model_input=",
    "degrees_rotate=",
    "Resize",
]

indices = set(range(min(65, len(linhas))))

for i, linha in enumerate(linhas):
    if any(termo in linha for termo in termos):
        indices.update(
            range(
                max(0, i - 5),
                min(len(linhas), i + 9)
            )
        )

print("=== INTEGRAÇÃO ALP + SAM EXISTENTE ===")
print("Arquivo:", ARQUIVO.relative_to(REPO))

anterior = -2

for i in sorted(indices):
    if i != anterior + 1:
        print("\n" + "-" * 55)

    print(f"{i+1:04}: {linhas[i]}")
    anterior = i

In [ ]:
from pathlib import Path
import ast
import json

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
ARQUIVO = REPO / "models/ProtoSAM.py"

codigo = ARQUIVO.read_text(encoding="utf-8")
linhas = codigo.splitlines()
arvore = ast.parse(codigo)

print("=== IMPLEMENTAÇÃO PROTOSAM ===")

for classe in arvore.body:
    if not isinstance(classe, ast.ClassDef):
        continue

    if classe.name not in ["ProtoSAM", "ALPNetWrapper"]:
        continue

    print(f"\nCLASSE: {classe.name}")

    for metodo in classe.body:
        if not isinstance(metodo, ast.FunctionDef):
            continue

        if metodo.name not in ["__init__", "forward"]:
            continue

        print(f"\n--- {metodo.name} ---")

        inicio = metodo.lineno - 1
        fim = metodo.end_lineno

        # Mostrar inicialização e os pontos relevantes do forward
        indices = set(range(
            inicio,
            min(inicio + 20, fim)
        ))

        termos = [
            "resize", "interpolate", "preprocess",
            "postprocess", "1024", "image_size",
            "input_size", "sam.", "prompt",
            "query_image", "coarse_model",
            "sam_model", "transform"
        ]

        for i in range(inicio, fim):
            if any(t in linhas[i].lower() for t in termos):
                indices.update(
                    range(max(inicio, i - 3), min(fim, i + 5))
                )

        anterior = -2

        for i in sorted(indices):
            if i != anterior + 1:
                print("  ...")
            print(f"{i+1:04}: {linhas[i]}")
            anterior = i

print("\n=== CONFIGURAÇÃO SAM DA REFERÊNCIA ===")

config_path = (
    REPO / "reproducao/"
    "eft_tiny_passo_2000_validacao_20261011T221527472147Z/"
    "config.json"
)

if config_path.exists():
    config = json.loads(config_path.read_text())

    for chave in ["sam", "selecao_sam", "alp_inferencia"]:
        print(f"\n{chave}:")
        print(json.dumps(
            config.get(chave, "não encontrado"),
            indent=2,
            ensure_ascii=False
        ))
else:
    print("Configuração histórica não encontrada.")

print("\n=== PESOS SAM ===")
pesos = REPO / "pretrained_model/sam_vit_h.pth"
print("SAM-H disponível:", pesos.is_file())

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import json
import sys
import os

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = REPO / "exp/eft_ct_piloto200_v1/avaliar_alp_sam_ct.py"

RUN = (
    REPO / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"
    / "avaliacao_alp_sam_rins_f0_4ckpts"
)

codigo = r'''
import os
import sys
import csv
import json
import time
import traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import SimpleITK as sitk

ROOT = Path(__file__).resolve().parents[2]
MODULOS = ROOT / "exp/eft_tiny_512_v1"
DADOS = ROOT / "data/SABS/sabs_CT_normalized"

PILOTO = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
TREINO = ROOT / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"

REFERENCIA_ALP = (
    TREINO / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
)

SAIDA = TREINO / "avaliacao_alp_sam_rins_f0_4ckpts"

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(MODULOS))

from eft_modelo_672 import construir_modelo
from models.ProtoSAM import ProtoSAM, ALPNetWrapper, ALPNetInput

PASSOS = [0, 2000, 5000, 10000]
ORGAOS = [(2, "KID_R"), (3, "KID_l")]
QUERIES = list(range(6))

torch.backends.cuda.matmul.allow_tf32 = False


def salvar_json(arquivo, obj):
    temp = arquivo.with_suffix(".json.tmp")
    temp.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(temp, arquivo)


def checkpoint(passo):
    if passo == 0:
        return PILOTO / "checkpoint_0000.pt"
    return TREINO / f"checkpoint_{passo:05d}.pt"


def volume(pid):
    img = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
    )
    gt = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"label_{pid}.nii.gz"))
    )
    assert img.shape == gt.shape
    assert img.shape[1:] == (256, 256)
    return img, gt


def preparar(fatia):
    rgb = np.repeat(
        fatia.astype(np.float32)[..., None], 3, axis=2
    )

    t = torch.from_numpy(
        np.ascontiguousarray(rgb)
    ).permute(2, 0, 1)[None].cuda()

    media = t.new_tensor(
        [123.675, 116.28, 103.53]
    )[None, :, None, None]

    desvio = t.new_tensor(
        [58.395, 57.12, 57.375]
    )[None, :, None, None]

    return (t - media) / desvio


def metricas(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())

    return {
        "dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "iou": tp / max(tp + fp + fn, 1),
        "precisao": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "pixels_pred": int(pred.sum())
    }


def main():
    assert torch.cuda.is_available()
    SAIDA.mkdir(parents=True, exist_ok=True)

    destino = SAIDA / "metricas_fatias.csv"
    assert not destino.exists(), "Avaliação já existe."

    for passo in PASSOS:
        assert checkpoint(passo).is_file()

    # Referência ALP já avaliada, nos mesmos pacientes.
    referencia = {}

    with REFERENCIA_ALP.open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            chave = (
                int(r["checkpoint"]),
                r["orgao"],
                int(r["paciente"]),
                int(r["fatia"])
            )
            assert chave not in referencia
            referencia[chave] = r

    mapa = json.loads(
        (DADOS / "classmap_1.json").read_text()
    )

    # Carregar suporte fixo para os dois rins.
    img_s, gt_s = volume(6)

    suportes = {}

    for classe, nome in ORGAOS:
        fatias = np.flatnonzero(
            np.any(gt_s == classe, axis=(1, 2))
        )
        assert len(fatias) > 0
        z = int(fatias[len(fatias) // 2])

        sup_img = preparar(img_s[z])
        sup_mask = torch.from_numpy(
            (gt_s[z] == classe).astype(np.float32)
        )[None].cuda()

        suportes[classe] = (sup_img, sup_mask, z)
        print(f"Suporte {nome}: paciente 6, fatia {z}", flush=True)

    # Configuração SAM-H validada no teste anterior.
    cfg_path = (
        ROOT / "reproducao"
        / "eft_tiny_passo_2000_validacao_20261011T221527472147Z"
        / "config.json"
    )

    cfg = dict(json.loads(cfg_path.read_text())["sam"])
    cfg["image_size"] = [1024, 1024]
    cfg["alpnet_image_size"] = 256
    cfg["sam_pretrained_path"] = str(
        ROOT / "pretrained_model/sam_vit_h.pth"
    )

    assert cfg["use_bbox"] and cfg["use_points"]
    assert not cfg["use_mask"]
    assert Path(cfg["sam_pretrained_path"]).is_file()

    print("Construindo VMamba-Tiny e SAM-H...", flush=True)

    modelo = construir_modelo("tiny", repo=ROOT)
    modelo.cuda().eval().requires_grad_(False)

    wrapper = ALPNetWrapper(modelo)

    protosam = ProtoSAM(
        coarse_segmentation_model=wrapper,
        **cfg
    ).to("cuda", dtype=torch.float32)

    protosam.eval().requires_grad_(False)

    total = 425 * len(PASSOS)
    concluidos = 0
    inicio = time.perf_counter()

    campos = [
        "checkpoint", "orgao", "paciente", "fatia",
        "dice_alp", "dice_sam", "iou_sam",
        "precisao_sam", "recall_sam",
        "tp_sam", "fp_sam", "fn_sam",
        "pixels_gt", "pixels_sam"
    ]

    registros = []

    with destino.open(
        "x", newline="", encoding="utf-8", buffering=1
    ) as f:
        escritor = csv.DictWriter(f, fieldnames=campos)
        escritor.writeheader()

        for passo in PASSOS:
            ckpt = torch.load(
                checkpoint(passo),
                map_location="cpu",
                weights_only=False
            )

            assert ckpt["passo"] == passo
            modelo.load_state_dict(
                ckpt["modelo"], strict=True
            )
            del ckpt

            modelo.eval()
            protosam.eval()

            print(f"\nCHECKPOINT {passo}", flush=True)

            for classe, nome in ORGAOS:
                sup_img, sup_mask, _ = suportes[classe]

                for pid in QUERIES:
                    imagens, labels = volume(pid)

                    fatias = mapa[nome][str(pid)]

                    for z in fatias:
                        z = int(z)
                        gt = labels[z] == classe
                        assert gt.any()

                        chave = (passo, nome, pid, z)
                        assert chave in referencia, chave

                        query = preparar(imagens[z])

                        def entrada():
                            return ALPNetInput(
                                support_images=[sup_img],
                                support_labels=[sup_mask],
                                query_images=query,
                                isval=True,
                                val_wsize=2,
                                show_viz=False,
                                supp_fts=None
                            )

                        with (
                            torch.no_grad(),
                            torch.autocast("cuda", enabled=False)
                        ):
                            pred_sam, scores = protosam(
                                query_image=query,
                                coarse_model_input=entrada(),
                                degrees_rotate=0
                            )

                            pred_sam = (
                                pred_sam.detach()
                                .cpu().numpy().squeeze()
                            )

                        assert pred_sam.shape == (256, 256)
                        assert np.isin(pred_sam, [0, 1]).all()

                        m = metricas(pred_sam.astype(bool), gt)

                        r = {
                            "checkpoint": passo,
                            "orgao": nome,
                            "paciente": pid,
                            "fatia": z,
                            "dice_alp": float(
                                referencia[chave]["dice"]
                            ),
                            "dice_sam": m["dice"],
                            "iou_sam": m["iou"],
                            "precisao_sam": m["precisao"],
                            "recall_sam": m["recall"],
                            "tp_sam": m["tp"],
                            "fp_sam": m["fp"],
                            "fn_sam": m["fn"],
                            "pixels_gt": int(gt.sum()),
                            "pixels_sam": m["pixels_pred"]
                        }

                        escritor.writerow(r)
                        registros.append(r)
                        concluidos += 1

                        if concluidos % 25 == 0:
                            salvar_json(
                                SAIDA / "status.json",
                                {
                                    "estado": "executando",
                                    "concluidos": concluidos,
                                    "total": total,
                                    "checkpoint": passo,
                                    "orgao": nome,
                                    "paciente": pid,
                                    "segundos": time.perf_counter() - inicio
                                }
                            )
                            print(
                                f"{concluidos}/{total}",
                                flush=True
                            )

                        del query, pred_sam, scores

    assert concluidos == total

    # Agregação por paciente.
    grupos = defaultdict(list)

    for r in registros:
        chave = (
            r["checkpoint"],
            r["orgao"],
            r["paciente"]
        )
        grupos[chave].append(r)

    pacientes = []

    for (passo, orgao, pid), itens in sorted(grupos.items()):
        pacientes.append({
            "checkpoint": passo,
            "orgao": orgao,
            "paciente": pid,
            "n_fatias": len(itens),
            "dice_alp": float(np.mean([
                r["dice_alp"] for r in itens
            ])),
            "dice_sam": float(np.mean([
                r["dice_sam"] for r in itens
            ]))
        })

    with (SAIDA / "resumo_pacientes.csv").open(
        "w", newline="", encoding="utf-8"
    ) as f:
        escritor = csv.DictWriter(
            f, fieldnames=list(pacientes[0])
        )
        escritor.writeheader()
        escritor.writerows(pacientes)

    # Média e desvio padrão entre os 6 pacientes.
    resumo = []

    print("\n=== RESULTADOS POR ÓRGÃO ===", flush=True)

    for passo in PASSOS:
        for _, orgao in ORGAOS:
            itens = [
                p for p in pacientes
                if p["checkpoint"] == passo
                and p["orgao"] == orgao
            ]
            assert len(itens) == 6

            alp = np.array([p["dice_alp"] for p in itens])
            sam = np.array([p["dice_sam"] for p in itens])

            linha = {
                "checkpoint": passo,
                "orgao": orgao,
                "n_pacientes": 6,
                "dice_alp_media": float(alp.mean()),
                "dice_alp_dp": float(alp.std(ddof=1)),
                "dice_sam_media": float(sam.mean()),
                "dice_sam_dp": float(sam.std(ddof=1)),
                "delta_sam_pp": float(
                    (sam.mean() - alp.mean()) * 100
                )
            }

            resumo.append(linha)

            print(
                f"{orgao} | passo {passo} | "
                f"ALP {alp.mean():.4f} ± {alp.std(ddof=1):.4f} | "
                f"SAM {sam.mean():.4f} ± {sam.std(ddof=1):.4f}",
                flush=True
            )

    with (SAIDA / "resumo_geral.csv").open(
        "w", newline="", encoding="utf-8"
    ) as f:
        escritor = csv.DictWriter(
            f, fieldnames=list(resumo[0])
        )
        escritor.writeheader()
        escritor.writerows(resumo)

    salvar_json(
        SAIDA / "status.json",
        {
            "estado": "concluido",
            "concluidos": concluidos,
            "total": total,
            "segundos": time.perf_counter() - inicio
        }
    )

    print("\nAVALIAÇÃO ALP + SAM CONCLUÍDA", flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception:
        erro = traceback.format_exc()
        print(erro, flush=True)

        SAIDA.mkdir(parents=True, exist_ok=True)
        salvar_json(
            SAIDA / "status.json",
            {
                "estado": "erro",
                "traceback": erro
            }
        )
        raise
'''

# Conferir sintaxe antes de criar os arquivos.
compile(codigo, str(SCRIPT), "exec")

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "Script existente é diferente. Não sobrescrito."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

# Conferir pré-requisitos.
PILOTO = REPO / "reproducao/eft_ct_felzenszwalb_fold0_piloto200"
TREINO = REPO / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"

for passo in [0, 2000, 5000, 10000]:
    arquivo = (
        PILOTO / "checkpoint_0000.pt"
        if passo == 0
        else TREINO / f"checkpoint_{passo:05d}.pt"
    )
    assert arquivo.is_file(), arquivo

assert (
    TREINO / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
).is_file()

assert (
    REPO / "pretrained_model/sam_vit_h.pth"
).is_file()

# Iniciar execução detached.
RUN.mkdir(parents=True, exist_ok=True)

LOG = RUN / "console.log"
LANCAMENTO = RUN / "lancamento.json"

for arquivo in [
    LANCAMENTO,
    LOG,
    RUN / "status.json",
    RUN / "metricas_fatias.csv"
]:
    assert not arquivo.exists(), (
        f"Execução anterior encontrada: {arquivo}"
    )

fd = os.open(
    str(LANCAMENTO),
    os.O_CREAT | os.O_EXCL | os.O_WRONLY,
    0o600
)

try:
    ambiente = os.environ.copy()
    ambiente["PYTHONUNBUFFERED"] = "1"

    with LOG.open("x", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(REPO),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=ambiente
        )

    registro = {
        "pid": processo.pid,
        "python": sys.executable,
        "script": str(SCRIPT),
        "script_sha256": hashlib.sha256(
            SCRIPT.read_bytes()
        ).hexdigest(),
        "inicio_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "checkpoints": [0, 2000, 5000, 10000],
        "inferencias_previstas": 1700,
        "modo": "detached"
    }

    with os.fdopen(fd, "w", encoding="utf-8") as f:
        json.dump(registro, f, indent=2)

except Exception:
    try:
        os.close(fd)
    except OSError:
        pass
    LANCAMENTO.unlink(missing_ok=True)
    raise

print("=== AVALIAÇÃO ALP + SAM LANÇADA ===")
print("PID:", processo.pid)
print("Script:", SCRIPT.name)
print("Resultados:", RUN)
print("Checkpoints:", [0, 2000, 5000, 10000])
print("Inferências previstas: 1700")
print("Estado: lançamento solicitado; execução ainda não verificada")

In [ ]:
from pathlib import Path
from collections import Counter, deque
import csv
import json
import subprocess

RUN = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/"
    "reproducao/eft_ct_felzenszwalb_fold0_10000_v1/"
    "avaliacao_alp_sam_rins_f0_4ckpts"
)

CHECKPOINTS = [0, 2000, 5000, 10000]
TOTAL = 1700
POR_CHECKPOINT = 425

def ler_json(arquivo):
    try:
        return json.loads(arquivo.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return {}

def processo_ativo(pid):
    if not pid:
        return False

    proc = Path(f"/proc/{pid}")

    try:
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        return (
            b"avaliar_alp_sam_ct.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        return False

lancamento = ler_json(RUN / "lancamento.json")
status = ler_json(RUN / "status.json")

pid = lancamento.get("pid")
ativo = processo_ativo(pid)

# Ler resultados registrados até o momento
contagem = Counter()
ultimos = deque(maxlen=5)
concluidas = 0

arquivo = RUN / "metricas_fatias.csv"

if arquivo.exists():
    with arquivo.open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            try:
                contagem[int(r["checkpoint"])] += 1
                ultimos.append(r)
                concluidas += 1
            except (KeyError, ValueError):
                pass

percentual = 100 * concluidas / TOTAL

print("=" * 68)
print("AVALIAÇÃO EFT-CT | ALP + SAM-H | 4 CHECKPOINTS")
print("=" * 68)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))

print(f"\nPROGRESSO: {concluidas:,}/{TOTAL:,} ({percentual:.2f}%)")

blocos = min(30, int(percentual * 30 / 100))
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

print("\n=== PROGRESSO POR CHECKPOINT ===")
print("Checkpoint | Concluídas | Total | Progresso")
print("-" * 54)

for passo in CHECKPOINTS:
    n = contagem[passo]

    print(
        f"{passo:10d} | "
        f"{n:10d} | "
        f"{POR_CHECKPOINT:5d} | "
        f"{100 * n / POR_CHECKPOINT:6.1f}%"
    )

if ultimos:
    print("\n=== ÚLTIMAS 5 INFERÊNCIAS ===")
    print("Ckpt  | Órgão | Paciente | Fatia | ALP    | SAM")
    print("-" * 60)

    for r in ultimos:
        print(
            f"{int(r['checkpoint']):5d} | "
            f"{r['orgao']:5s} | "
            f"{int(r['paciente']):8d} | "
            f"{int(r['fatia']):5d} | "
            f"{float(r['dice_alp']):.4f} | "
            f"{float(r['dice_sam']):.4f}"
        )

# Mostrar os resultados quando o resumo estiver disponível
resumo = RUN / "resumo_geral.csv"

if resumo.exists():
    print("\n=== RESULTADOS FINAIS — DICE MÉDIO ± DP ===")

    with resumo.open(newline="", encoding="utf-8") as f:
        linhas = list(csv.DictReader(f))

    print(
        "Checkpoint | Órgão | ALP             | "
        "ALP + SAM       | Δ SAM (p.p.)"
    )
    print("-" * 79)

    for passo in CHECKPOINTS:
        for orgao in ["KID_R", "KID_l"]:
            encontrados = [
                r for r in linhas
                if int(r["checkpoint"]) == passo
                and r["orgao"] == orgao
            ]

            if len(encontrados) != 1:
                continue

            r = encontrados[0]

            alp = (
                f"{float(r['dice_alp_media']):.4f} ± "
                f"{float(r['dice_alp_dp']):.4f}"
            )

            sam = (
                f"{float(r['dice_sam_media']):.4f} ± "
                f"{float(r['dice_sam_dp']):.4f}"
            )

            print(
                f"{passo:10d} | "
                f"{orgao:5s} | "
                f"{alp:15s} | "
                f"{sam:15s} | "
                f"{float(r['delta_sam_pp']):+7.2f}"
            )

print("\n=== GPU ===")

try:
    resultado = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,used_gpu_memory",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    processos_gpu = [
        linha.strip()
        for linha in resultado.stdout.splitlines()
        if linha.split(",")[0].strip() == str(pid)
    ]

    if processos_gpu:
        print("Processo na GPU:", processos_gpu[0], "MiB")
    else:
        print("PID não encontrado entre os processos CUDA")

except Exception as e:
    print("Consulta GPU indisponível:", e)

print("\n=== DIAGNÓSTICO ===")

estado = status.get("estado")

if estado == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "")[-3500:])

elif estado == "concluido":
    print("AVALIAÇÃO CONCLUÍDA")
    print("Inferências registradas:", concluidas)

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada")

console = RUN / "console.log"

if console.exists():
    print("\n=== ÚLTIMAS MENSAGENS ===")

    ultimas_linhas = deque(maxlen=15)

    with console.open(encoding="utf-8", errors="replace") as f:
        for linha in f:
            ultimas_linhas.append(linha.rstrip())

    for linha in ultimas_linhas:
        print(linha)

In [ ]:
from pathlib import Path
import csv
import json

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

RUN = (
    REPO / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"
    / "avaliacao_alp_sam_rins_f0_4ckpts"
)

DADOS = REPO / "data/SABS/sabs_CT_normalized"

# 1. Identificar a próxima inferência na ordem original
mapa = json.loads(
    (DADOS / "classmap_1.json").read_text()
)

esperadas = [
    (passo, orgao, pid, int(z))
    for passo in [0, 2000, 5000, 10000]
    for orgao in ["KID_R", "KID_l"]
    for pid in range(6)
    for z in mapa[orgao][str(pid)]
]

with (RUN / "metricas_fatias.csv").open(
    newline="", encoding="utf-8"
) as f:
    concluidas = list(csv.DictReader(f))

assert len(esperadas) == 1700

# Conferir se o CSV é um prefixo completo,
# sem lacunas ou inferências fora de ordem.
for indice, registro in enumerate(concluidas):
    chave = (
        int(registro["checkpoint"]),
        registro["orgao"],
        int(registro["paciente"]),
        int(registro["fatia"])
    )

    assert chave == esperadas[indice], (
        f"Divergência na linha {indice + 1}: "
        f"{chave} != {esperadas[indice]}"
    )

print("=== ESTADO DA AVALIAÇÃO ===")
print("Inferências concluídas:", len(concluidas))
print("Inferências restantes:", len(esperadas) - len(concluidas))
print("CSV íntegro e sequencial: SIM")

print("\nÚltima inferência concluída:")
print(esperadas[len(concluidas) - 1])

print("\nPróxima inferência (que falhou):")
print(esperadas[len(concluidas)])

# 2. Inspecionar os caminhos de retorno do ProtoSAM
arquivo = REPO / "models/ProtoSAM.py"
linhas = arquivo.read_text(encoding="utf-8").splitlines()

print("\n=== CAMINHOS DE RETORNO DO PROTOSAM ===")

for inicio, fim in [(550, 620), (650, 680)]:
    print(f"\n--- Linhas {inicio}–{fim} ---")
    for numero in range(inicio, min(fim, len(linhas)) + 1):
        print(f"{numero:04d}: {linhas[numero - 1]}")

print("\nNenhum arquivo modificado.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import subprocess
import sys

REPO = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ORIGINAL = (
    REPO / "exp/eft_ct_piloto200_v1/avaliar_alp_sam_ct.py"
)

CORRIGIDO = (
    REPO / "exp/eft_ct_piloto200_v1/"
    "avaliar_alp_sam_ct_retomada.py"
)

RUN = (
    REPO / "reproducao/eft_ct_felzenszwalb_fold0_10000_v1"
    / "avaliacao_alp_sam_rins_f0_4ckpts"
)

CSV = RUN / "metricas_fatias.csv"
LOG = RUN / "console_retomada.log"
LANCAMENTO = RUN / "lancamento_retomada.json"

# Confirmar estado anterior
assert ORIGINAL.is_file()
assert CSV.is_file()

with CSV.open(newline="", encoding="utf-8") as f:
    anteriores = list(csv.DictReader(f))

assert len(anteriores) == 478, (
    f"Esperadas 478 inferências; encontradas {len(anteriores)}"
)

# Confirmar que o código original não mudou
registro_original = json.loads(
    (RUN / "lancamento.json").read_text()
)

digest_original = hashlib.sha256(
    ORIGINAL.read_bytes()
).hexdigest()

assert digest_original == registro_original["script_sha256"]

codigo = ORIGINAL.read_text(encoding="utf-8")

def substituir(antigo, novo):
    global codigo

    quantidade = codigo.count(antigo)

    assert quantidade == 1, (
        f"Trecho esperado não encontrado exatamente uma vez "
        f"({quantidade}): {antigo[:100]}"
    )

    codigo = codigo.replace(antigo, novo, 1)


# 1. Permitir continuar usando o CSV existente
substituir(
    'assert not destino.exists(), "Avaliação já existe."',
    'assert destino.exists(), "CSV anterior não encontrado."'
)

# 2. Conferir a integridade das 478 linhas
marcador = '''    mapa = json.loads(
        (DADOS / "classmap_1.json").read_text()
    )'''

substituir(
    marcador,
    marcador + '''

    # Retomada: verificar o prefixo já concluído.
    esperadas = [
        (passo, nome, pid, int(z))
        for passo in PASSOS
        for _, nome in ORGAOS
        for pid in QUERIES
        for z in mapa[nome][str(pid)]
    ]

    assert len(esperadas) == 1700

    with destino.open(newline="", encoding="utf-8") as f:
        anteriores = list(csv.DictReader(f))

    assert len(anteriores) == 478

    for indice, r in enumerate(anteriores):
        chave = (
            int(r["checkpoint"]),
            r["orgao"],
            int(r["paciente"]),
            int(r["fatia"])
        )

        assert chave == esperadas[indice], (
            f"CSV divergente na linha {indice + 1}"
        )

    ja_concluidas = set(esperadas[:len(anteriores)])

    print(
        f"Retomando: {len(anteriores)}/1700 concluídas; "
        f"{1700-len(anteriores)} restantes",
        flush=True
    )

    salvar_json(SAIDA / "status.json", {
        "estado": "retomando",
        "concluidos": len(anteriores),
        "total": 1700
    })
'''
)

# 3. Inicializar contadores com os resultados existentes
substituir(
    '    concluidos = 0\n    inicio = time.perf_counter()',
    '    concluidos = len(anteriores)\n    inicio = time.perf_counter()'
)

substituir(
    '    registros = []',
    '''    registros = [
        {
            **r,
            "checkpoint": int(r["checkpoint"]),
            "paciente": int(r["paciente"]),
            "dice_alp": float(r["dice_alp"]),
            "dice_sam": float(r["dice_sam"])
        }
        for r in anteriores
    ]'''
)

# 4. Acrescentar linhas ao CSV, sem reescrever o cabeçalho
substituir(
    '''    with destino.open(
        "x", newline="", encoding="utf-8", buffering=1
    ) as f:
        escritor = csv.DictWriter(f, fieldnames=campos)
        escritor.writeheader()''',
    '''    with destino.open(
        "a", newline="", encoding="utf-8", buffering=1
    ) as f:
        escritor = csv.DictWriter(f, fieldnames=campos)'''
)

# 5. Ignorar todas as inferências já concluídas
substituir(
    '''                        chave = (passo, nome, pid, z)
                        assert chave in referencia, chave

                        query = preparar(imagens[z])''',
    '''                        chave = (passo, nome, pid, z)
                        assert chave in referencia, chave

                        if chave in ja_concluidas:
                            continue

                        query = preparar(imagens[z])'''
)

# 6. Corrigir somente a saída antecipada sem foreground
substituir(
    '''                        assert pred_sam.shape == (256, 256)
                        assert np.isin(pred_sam, [0, 1]).all()''',
    '''                        if pred_sam.shape == (1024, 1024):
                            # ProtoSAM retornou antecipadamente
                            # porque o ALP não encontrou foreground.
                            assert not np.any(pred_sam), (
                                "Máscara 1024 inesperadamente não vazia."
                            )

                            # Registrar que SAM não foi executado.
                            with (
                                SAIDA / "sem_prompt_sam.jsonl"
                            ).open("a", encoding="utf-8") as log_f:
                                log_f.write(json.dumps({
                                    "checkpoint": passo,
                                    "orgao": nome,
                                    "paciente": pid,
                                    "fatia": z,
                                    "motivo": "ALP sem foreground",
                                    "sam_executado": False
                                }) + "\\n")

                            pred_sam = np.zeros(
                                (256, 256), dtype=np.uint8
                            )

                            print(
                                f"Sem foreground: checkpoint={passo}, "
                                f"orgao={nome}, paciente={pid}, "
                                f"fatia={z}",
                                flush=True
                            )

                        assert pred_sam.shape == (256, 256), (
                            f"Formato inesperado: {pred_sam.shape}"
                        )
                        assert np.isin(pred_sam, [0, 1]).all()'''
)

# 7. Verificar quantidade antes de agregar métricas
substituir(
    '    assert concluidos == total',
    '''    assert concluidos == total
    assert len(registros) == total'''
)

# Validar sintaxe
compile(codigo, str(CORRIGIDO), "exec")

# Preservar scripts existentes
if CORRIGIDO.exists():
    assert CORRIGIDO.read_text(encoding="utf-8") == codigo, (
        "Existe uma versão corrigida diferente."
    )
else:
    CORRIGIDO.write_text(codigo, encoding="utf-8")

# Impedir lançamentos duplicados
assert not LANCAMENTO.exists()
assert not LOG.exists()
assert not (RUN / "resumo_geral.csv").exists()

fd = os.open(
    str(LANCAMENTO),
    os.O_CREAT | os.O_EXCL | os.O_WRONLY,
    0o600
)

try:
    ambiente = os.environ.copy()
    ambiente["PYTHONUNBUFFERED"] = "1"

    with LOG.open("x", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(CORRIGIDO)],
            cwd=str(REPO),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=ambiente
        )

    registro = {
        "pid": processo.pid,
        "inicio_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "script": str(CORRIGIDO),
        "script_sha256": hashlib.sha256(
            CORRIGIDO.read_bytes()
        ).hexdigest(),
        "inferencias_preservadas": 478,
        "inferencias_restantes": 1222,
        "modo": "detached"
    }

    with os.fdopen(fd, "w", encoding="utf-8") as f:
        json.dump(registro, f, indent=2)

except Exception:
    try:
        os.close(fd)
    except OSError:
        pass
    LANCAMENTO.unlink(missing_ok=True)
    raise

print("=== RETOMADA ALP + SAM ===")
print("PID:", processo.pid)
print("Inferências preservadas:", len(anteriores))
print("Inferências restantes:", 1700 - len(anteriores))
print("Próxima inferência: 479")
print("Script:", CORRIGIDO.name)
print("Log:", LOG.name)
print("Modo: detached")
print("Lançamento solicitado; execução ainda não verificada")

In [ ]:
from pathlib import Path
from collections import Counter, deque
import csv
import json
import subprocess

RUN = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/"
    "reproducao/eft_ct_felzenszwalb_fold0_10000_v1/"
    "avaliacao_alp_sam_rins_f0_4ckpts"
)

CHECKPOINTS = [0, 2000, 5000, 10000]
TOTAL = 1700
PRESERVADAS = 478

def ler_json(caminho):
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return {}

def ativo(pid):
    if not pid:
        return False

    proc = Path(f"/proc/{pid}")

    try:
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        return (
            b"avaliar_alp_sam_ct_retomada.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        return False

lancamento = ler_json(RUN / "lancamento_retomada.json")
status = ler_json(RUN / "status.json")

pid = lancamento.get("pid")
em_execucao = ativo(pid)

# Ler CSV existente + inferências da retomada
contagem = Counter()
ultimos = deque(maxlen=5)
concluidas = 0

csv_path = RUN / "metricas_fatias.csv"

if csv_path.exists():
    with csv_path.open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            try:
                contagem[int(r["checkpoint"])] += 1
                ultimos.append(r)
                concluidas += 1
            except (KeyError, ValueError):
                pass

novas = max(0, concluidas - PRESERVADAS)
restantes = max(0, TOTAL - concluidas)
percentual = min(100, 100 * concluidas / TOTAL)

print("=" * 67)
print("AVALIAÇÃO ALP + SAM | RETOMADA | VMamba-Tiny CT")
print("=" * 67)

print("PID da retomada:", pid)
print("Processo ativo:", "SIM" if em_execucao else "NÃO")
print("Estado registrado:", status.get("estado", "desconhecido"))

print(f"\nPROGRESSO TOTAL: {concluidas:,}/{TOTAL:,} ({percentual:.2f}%)")
print(f"Inferências preservadas: {PRESERVADAS}")
print(f"Novas inferências: {novas:,}/1222")
print(f"Restantes: {restantes:,}")

blocos = min(30, int(percentual * 30 / 100))
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

print("\n=== PROGRESSO POR CHECKPOINT ===")
print("Checkpoint | Concluídas | Total | Progresso")
print("-" * 53)

for passo in CHECKPOINTS:
    n = contagem[passo]
    print(
        f"{passo:10d} | "
        f"{n:10d} | "
        f"{425:5d} | "
        f"{100*n/425:6.1f}%"
    )

if ultimos:
    print("\n=== ÚLTIMAS 5 INFERÊNCIAS ===")
    print("Ckpt  | Órgão | Paciente | Fatia | Dice ALP | Dice SAM")
    print("-" * 67)

    for r in ultimos:
        print(
            f"{int(r['checkpoint']):5d} | "
            f"{r['orgao']:5s} | "
            f"{int(r['paciente']):8d} | "
            f"{int(r['fatia']):5d} | "
            f"{float(r['dice_alp']):8.4f} | "
            f"{float(r['dice_sam']):8.4f}"
        )

# Contabilizar episódios sem foreground
sem_prompt = RUN / "sem_prompt_sam.jsonl"
sem_prompt_por_ckpt = Counter()
sem_prompt_total = 0

if sem_prompt.exists():
    with sem_prompt.open(encoding="utf-8") as f:
        for linha in f:
            try:
                r = json.loads(linha)
                sem_prompt_por_ckpt[int(r["checkpoint"])] += 1
                sem_prompt_total += 1
            except (ValueError, KeyError):
                pass

print("\n=== CASOS SEM PROMPT PARA O SAM ===")
print("Total registrado:", sem_prompt_total)

for passo in CHECKPOINTS:
    print(
        f"Checkpoint {passo:5d}: "
        f"{sem_prompt_por_ckpt[passo]} casos"
    )

# Resultados finais, caso a avaliação tenha terminado
resumo = RUN / "resumo_geral.csv"

if resumo.exists():
    print("\n=== DICE MÉDIO ± DESVIO PADRÃO ENTRE PACIENTES ===")

    with resumo.open(newline="", encoding="utf-8") as f:
        linhas = list(csv.DictReader(f))

    print("Passo | Órgão | ALP            | ALP + SAM      | Δ p.p.")
    print("-" * 71)

    for passo in CHECKPOINTS:
        for orgao in ["KID_R", "KID_l"]:
            encontrados = [
                r for r in linhas
                if int(r["checkpoint"]) == passo
                and r["orgao"] == orgao
            ]

            if len(encontrados) != 1:
                continue

            r = encontrados[0]

            print(
                f"{passo:5d} | "
                f"{orgao:5s} | "
                f"{float(r['dice_alp_media']):.4f} ± "
                f"{float(r['dice_alp_dp']):.4f} | "
                f"{float(r['dice_sam_media']):.4f} ± "
                f"{float(r['dice_sam_dp']):.4f} | "
                f"{float(r['delta_sam_pp']):+6.2f}"
            )

# GPU
print("\n=== GPU ===")

try:
    resultado = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,used_gpu_memory",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    processos = [
        linha.strip()
        for linha in resultado.stdout.splitlines()
        if linha.split(",")[0].strip() == str(pid)
    ]

    print(
        "Processo na GPU: " + processos[0] + " MiB"
        if processos
        else "PID não encontrado na GPU"
    )

except Exception as e:
    print("Consulta indisponível:", e)

# Diagnóstico
print("\n=== DIAGNÓSTICO ===")

estado = status.get("estado")

if estado == "concluido" and concluidas == TOTAL:
    print("AVALIAÇÃO CONCLUÍDA — 1700/1700")

elif em_execucao:
    print("RETOMADA EM EXECUÇÃO")

elif estado == "erro":
    print("ERRO NA RETOMADA")
    print(status.get("traceback", "")[-3000:])

else:
    print("Processo inativo sem conclusão confirmada")

# Ler exclusivamente o console da retomada
log = RUN / "console_retomada.log"

if log.exists():
    print("\n=== ÚLTIMAS MENSAGENS DA RETOMADA ===")

    ultimas = deque(maxlen=15)

    with log.open(encoding="utf-8", errors="replace") as f:
        for linha in f:
            ultimas.append(linha.rstrip())

    for linha in ultimas:
        print(linha)